# ⚙️♟️ Solution (S) — le moteur C++ avec P.O.W.N. contre Stockfish

**Solution (S)** est le moteur d'échecs complet écrit en **C++** : générateur de coups (vérifié contre `python-chess` et les nombres officiels de *perft*),
réseau **P.O.W.N.** avec mise à jour incrémentale, recherche alpha-bêta améliorée (approfondissement itératif, table de transposition, coup nul,
réductions, élagages, quiescence avec SEE), gestion du temps, protocole UCI.

Ce notebook :
1. **compile** le moteur (les sources sont dans la cellule suivante, rien à télécharger) ;
2. **convertit** `pown.onnx` en `pown.bin` (poids pour le C++) ;
3. **teste** : règles (perft), vitesse, une recherche ;
4. fait jouer **POWN@S1** (1 seconde par coup) contre Stockfish 17 à plusieurs niveaux, comme les notebooks 07 et 08.

Repères : **PWN@ab3 ≈ 2264 Elo** (notebook 07). ⚠️ L'`UCI_Elo` de Stockfish est calibré contre des moteurs : c'est un ordre de grandeur, pas un Elo FIDE.

## ⚙️ Avant de lancer
1. Dataset **`pown-1`** (contenant `pown.onnx`) en **Input**.
2. **Internet ON**, accélérateur **None**, puis **Save & Run All (Commit)**.

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/POWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import json
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "POWN-soluce/pawn_soluce.onnx",
    "POWN": "Pown/pown.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


GPU_ID = 0       # carte utilisée par ce processus (Kaggle « GPU T4 x2 » en a deux)
GPU_MEM_MO = 0   # plafond mémoire GPU par réseau (0 = pas de plafond) ; fixé quand plusieurs processus partagent le GPU
_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        # HEURISTIC : cuDNN choisit ses algos sans essayer ceux qui demandent d'énormes
        # zones de travail (sinon des blocs de 150 Mo dépassent le plafond mémoire)
        opts = {"arena_extend_strategy": "kSameAsRequested", "cudnn_conv_algo_search": "HEURISTIC",
                "cudnn_conv_use_max_workspace": "0", "device_id": GPU_ID}
        if GPU_MEM_MO:
            opts["gpu_mem_limit"] = GPU_MEM_MO * 1024 * 1024
        return [("CUDAExecutionProvider", opts), "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=prov)
        self.sur_gpu = "CUDAExecutionProvider" in self.session.get_providers()
        if prov[0] != "CPUExecutionProvider":
            _CUDA_OK = self.sur_gpu
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        if self.sur_gpu and GPU_MEM_MO:
            lot = min(lot, 512)  # petits lots quand la mémoire GPU est plafonnée
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            n = len(x)
            if self.sur_gpu and GPU_MEM_MO:
                # taille de lot arrondie à une puissance de 2 : peu de formes différentes, donc
                # onnxruntime réutilise ses blocs mémoire au lieu d'en allouer de nouveaux à chaque coup
                taille = max(64, 1 << (n - 1).bit_length())
                x = np.concatenate([x, np.zeros((taille - n, 69), np.uint8)]) if taille > n else x
            try:
                res.append(self.session.run(None, {"board": x})[0][:n])
            except Exception as e:
                if not self.sur_gpu:
                    raise
                # GPU plein : ce réseau passe sur CPU pour la suite (mêmes résultats, plus lent)
                print(f"⚠️ {self.nom} : erreur GPU ({str(e)[:80]}...) -> CPU")
                self.session = ort.InferenceSession(poids(RESEAUX[self.nom]), self.so, providers=["CPUExecutionProvider"])
                self.sur_gpu = False
                res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- P.O.W.N. (NNUE HalfKAv2_hm) : autre encodage d'entrée, même interface -------------

def _bucket_pown(roi, miroir):
    f = chess.square_file(roi)
    if miroir:
        f = 7 - f
    return chess.square_rank(roi) * 4 + f


def features_pown(board, couleur):
    """doit rester identique à features() de Pown/commun_pown.py (vérifié par test)"""
    roi = board.king(couleur)
    miroir = chess.square_file(roi) >= 4
    b = _bucket_pown(roi, miroir)
    idx = []
    for case, piece in board.piece_map().items():
        if piece.piece_type == chess.KING and piece.color == couleur:
            continue
        f, r = chess.square_file(case), chess.square_rank(case)
        if miroir:
            f = 7 - f
        nous = piece.color == couleur
        t = (piece.piece_type - 1) if nous else 5 + (piece.piece_type - 1)
        idx.append(b * 704 + (r * 8 + f) * 11 + t)
    return idx


class EvaluateurPOWN(Evaluateur):
    """P.O.W.N. : ONNX à 4 entrées (index + décalages des deux points de vue) -> logit ; cp = 400 x logit"""

    def __init__(self, nom, gpu=False, threads=0):
        self.nom = nom
        self.so = ort.SessionOptions()
        if threads:
            self.so.intra_op_num_threads = threads
            self.so.inter_op_num_threads = 1
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), self.so, providers=["CPUExecutionProvider"])
        self.sur_gpu = False
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        res = []
        for i in range(0, len(fens), lot):
            idx_n, dec_n, idx_e, dec_e = [], [], [], []
            for fen in fens[i:i + lot]:
                b = chess.Board(fen)
                n, e = features_pown(b, b.turn), features_pown(b, not b.turn)
                dec_n.append(len(idx_n)); idx_n += n
                dec_e.append(len(idx_e)); idx_e += e
            sortie = self.session.run(None, {
                "idx_nous": np.array(idx_n, dtype=np.int64), "dec_nous": np.array(dec_n, dtype=np.int64),
                "idx_eux": np.array(idx_e, dtype=np.int64), "dec_eux": np.array(dec_e, dtype=np.int64)})[0]
            res.append(400.0 * sortie)
        self.nb_evals += len(fens)
        return np.concatenate(res)


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (POWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- recherche « équilibrée » : alpha-bêta améliorée + durée fixée par coup (POWN@t5 = 5 s par coup) ----

class _Temps(Exception):
    pass


class Recherche:
    """Alpha-bêta avec approfondissement itératif jusqu'à épuisement du temps. Par rapport à choisir_alphabeta :
    tri des coups (table, captures MVV-LVA, tueurs, historique), quiescence qui écarte les mauvaises captures,
    extension d'échec, coup nul, réductions (LMR). Le coup rendu est celui de la dernière profondeur terminée."""

    def __init__(self, ev, secondes, prof_max=20, prof_q=4):
        self.ev, self.secondes, self.prof_max, self.prof_q = ev, secondes, prof_max, prof_q
        self.tt, self.tueurs, self.histo, self.noeuds = {}, {}, {}, 0
        self.fin = 0.0
        self.prof_atteinte = 0

    def _horloge(self):
        self.noeuds += 1
        if self.noeuds & 15 == 0 and time.perf_counter() > self.fin:
            raise _Temps()

    def _cle(self, board, mv, coup_tt, ply):
        if mv == coup_tt:
            return 1_000_000
        if board.is_capture(mv) or mv.promotion:
            victime = VAL_PIECE[board.piece_type_at(mv.to_square) or chess.PAWN]
            return 100_000 + victime * 100 - VAL_PIECE[board.piece_type_at(mv.from_square)] + (50 if mv.promotion else 0)
        if mv in self.tueurs.get(ply, ()):
            return 90_000
        return self.histo.get((board.turn, mv.from_square, mv.to_square), 0)

    def quiescence(self, board, alpha, beta, prof):
        self._horloge()
        en_echec = board.is_check()
        if en_echec:
            coups = list(board.legal_moves)
            if not coups:
                return -MATE
            gain = -MATE
        else:
            gain = self.ev.eval1(board.fen())
            if gain >= beta:
                return gain
            alpha = max(alpha, gain)
            if prof <= 0:
                return gain
            coups = []
            for m in board.legal_moves:
                if board.is_capture(m) or m.promotion:
                    victime = VAL_PIECE[board.piece_type_at(m.to_square) or chess.PAWN]
                    # delta : même en gagnant la pièce, on reste loin de alpha
                    if gain + 100 * victime + 200 < alpha and not m.promotion:
                        continue
                    # mauvaise capture : pièce plus chère qui prend une pièce moins chère protégée
                    if (VAL_PIECE[board.piece_type_at(m.from_square)] > victime and not m.promotion
                            and board.is_attacked_by(not board.turn, m.to_square)):
                        continue
                    coups.append(m)
        coups.sort(key=lambda m: self._cle(board, m, None, 0), reverse=True)
        for mv in coups:
            board.push(mv)
            v = -self.quiescence(board, -beta, -alpha, prof - 1)
            board.pop()
            if v > gain:
                gain = v
                if v > alpha:
                    alpha = v
                    if alpha >= beta:
                        break
        return gain

    def negamax(self, board, depth, alpha, beta, ply, nul_ok=True):
        self._horloge()
        coups = list(board.legal_moves)
        if not coups:
            return -(MATE - ply) if board.is_check() else 0
        if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
            return 0
        alpha0 = alpha
        h = chess.polyglot.zobrist_hash(board)
        e = self.tt.get(h)
        coup_tt = e[3] if e else None
        if e and e[0] >= depth and e[2] != "none":
            _, v, drapeau, _ = e
            if drapeau == "exact":
                return v
            if drapeau == "min":
                alpha = max(alpha, v)
            else:
                beta = min(beta, v)
            if alpha >= beta:
                return v
        en_echec = board.is_check()
        if en_echec and ply < 40:
            depth += 1                       # extension d'échec
        if depth <= 0:
            return self.quiescence(board, alpha, beta, self.prof_q)
        # coup nul : si même en passant son tour on reste >= beta, on coupe (pas en finale de pions : zugzwang)
        if (nul_ok and depth >= 3 and not en_echec and beta < MATE - 1000
                and any(board.pieces(t, board.turn) for t in (chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN))):
            board.push(chess.Move.null())
            v = -self.negamax(board, depth - 3, -beta, -beta + 1, ply + 1, False)
            board.pop()
            if v >= beta:
                return beta
        coups.sort(key=lambda m: self._cle(board, m, coup_tt, ply), reverse=True)
        meilleur, meilleur_coup = -MATE - 1, None
        for i, mv in enumerate(coups):
            calme = not board.is_capture(mv) and not mv.promotion
            donne_echec = board.gives_check(mv)
            board.push(mv)
            if i >= 4 and depth >= 3 and calme and not en_echec and not donne_echec:
                v = -self.negamax(board, depth - 2, -alpha - 1, -alpha, ply + 1)      # réduit, fenêtre nulle
                if v > alpha:
                    v = -self.negamax(board, depth - 1, -beta, -alpha, ply + 1)
            else:
                v = -self.negamax(board, depth - 1, -beta, -alpha, ply + 1)
            board.pop()
            if v > meilleur:
                meilleur, meilleur_coup = v, mv
            if v > alpha:
                alpha = v
            if alpha >= beta:
                if calme:
                    t = self.tueurs.setdefault(ply, [])
                    if mv not in t:
                        t.insert(0, mv)
                        del t[2:]
                    k = (board.turn, mv.from_square, mv.to_square)
                    self.histo[k] = self.histo.get(k, 0) + depth * depth
                break
        if abs(meilleur) > MATE - 1000:
            drapeau = "none"                 # score de mat dépendant de la distance : on ne le réutilise pas
        else:
            drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
        self.tt[h] = (depth, meilleur, drapeau, meilleur_coup)
        return meilleur

    def choisir(self, board):
        b = board.copy(stack=True)
        coups = list(b.legal_moves)
        if len(coups) == 1:
            return coups[0]
        debut = time.perf_counter()
        self.fin = debut + self.secondes
        meilleur_coup, ordre = coups[0], coups
        for d in range(1, self.prof_max + 1):
            try:
                alpha, beta, meilleur_score, du_tour = -MATE - 1, MATE + 1, -MATE - 1, None
                for mv in ordre:
                    b.push(mv)
                    v = -self.negamax(b, d - 1, -beta, -alpha, 1)
                    b.pop()
                    if v > meilleur_score:
                        meilleur_score, du_tour = v, mv
                    alpha = max(alpha, v)
            except _Temps:
                break
            meilleur_coup, self.prof_atteinte = du_tour, d
            ordre = [du_tour] + [m for m in ordre if m != du_tour]    # le meilleur d'abord à la profondeur suivante
            if abs(meilleur_score) > MATE - 1000:
                break                                                # mat trouvé : inutile d'aller plus loin
            if time.perf_counter() - debut > 0.4 * self.secondes:
                break                                                # la profondeur suivante coûterait plus que ce qui reste
        return meilleur_coup


DERNIERE = {}   # profondeur atteinte et nombre de noeuds de la dernière recherche (pour les statistiques)


def choisir_temps(board, ev, secondes, prof_max=20):
    r = Recherche(ev, secondes, prof_max)
    mv = r.choisir(board)
    DERNIERE.update(profondeur=r.prof_atteinte, noeuds=r.noeuds)
    return mv


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "POWN@ab3" (POWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "POWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    if rech.startswith("S"):          # ex. "POWN@S1" : moteur C++ Solution (S), 1 s par coup
        return nom, "solution", float(rech[1:])
    if rech.startswith("e") and rech[1:].isdigit():   # ex. "POWN@e3" : profondeur fixe 3, recherche améliorée
        return nom, "econome", int(rech[1:])
    if rech.startswith("t"):          # ex. "POWN@t5" : recherche équilibrée, 5 secondes par coup
        return nom, "temps", float(rech[1:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom, gpu=None):
    gpu = GPU if gpu is None else gpu
    if (nom, gpu) not in _EVALS:
        classe = EvaluateurPOWN if nom == "POWN" else Evaluateur
        _EVALS[(nom, gpu)] = classe(nom, gpu=gpu, threads=THREADS)
    return _EVALS[(nom, gpu)]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()
    _S.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


_S = {}


def solution(secondes=1.0):
    """moteur C++ Solution (S) piloté en UCI ; chemins donnés par les variables d'environnement
    SOLUTION_BIN (exécutable) et SOLUTION_POIDS (pown.bin), lues à chaque appel (les processus fils les héritent)"""
    cle = (os.environ["SOLUTION_BIN"], os.environ["SOLUTION_POIDS"])
    if cle not in _S:
        e = chess.engine.SimpleEngine.popen_uci([cle[0], cle[1]])
        e.configure({"Hash": 64})
        _S[cle] = e
    return _S[cle]


def fermer_stockfish():
    for e in list(_SF.values()) + list(_S.values()):
        try:
            e.quit()
        except Exception:
            pass
    _SF.clear()
    _S.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, evaluateur(nom), depth)
    if rech == "solution":
        return solution(depth).play(b, chess.engine.Limit(time=depth)).move
    if rech == "econome":
        return choisir_temps(b, evaluateur(nom, gpu=False), 1e9, prof_max=depth)
    if rech == "temps":
        return choisir_temps(b, evaluateur(nom, gpu=False), depth)
    # l'alpha-bêta évalue une position à la fois : sur Kaggle la latence CPU (~6 ms)
    # est deux fois plus basse que celle du GPU (~14 ms), donc on reste sur CPU
    return choisir_alphabeta(b, evaluateur(nom, gpu=False), depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def rendre_executable(binaire):
    """/kaggle/working interdit d'exécuter des programmes : on copie Stockfish ailleurs
    et on garde le premier emplacement où il démarre vraiment"""
    for dest in ("/usr/local/bin/stockfish-pawn", "/tmp/stockfish-pawn", binaire):
        try:
            if dest != binaire:
                shutil.copy(binaire, dest)
            os.chmod(dest, 0o755)
            subprocess.run([dest, "quit"], check=True, timeout=10, capture_output=True)
            return dest
        except Exception as e:
            print("Stockfish ne démarre pas depuis", dest, ":", e)
    raise RuntimeError("Stockfish ne peut être exécuté nulle part")


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return rendre_executable(deja[0])
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            return rendre_executable(binaire)
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def ecrire_suivi(suivi, blancs, noirs, b, ouverture, debut):
    # position en cours dans un petit fichier JSON, lu par le notebook pour l'afficher en direct
    dernier = b.peek() if b.move_stack else None
    san = ""
    if dernier:
        prec = b.copy()
        prec.pop()
        san = prec.san(dernier)
    tmp = suivi + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"blancs": blancs, "noirs": noirs, "fen": b.fen(), "dernier": dernier.uci() if dernier else "",
                   "san": san, "ply": b.ply(), "ouverture": ouverture, "debut": debut}, f)
    os.replace(tmp, suivi)


def jouer_partie(blancs, noirs, ouverture, max_plies=300, suivi=None, nom_ouverture=""):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    positions = []  # chaque coup : qui, coup, position obtenue, temps de réflexion
    for u in ouverture:
        mv = chess.Move.from_uci(u)
        positions.append({"ply": b.ply() + 1, "joueur": "ouverture", "san": b.san(mv), "uci": u, "secondes": 0.0})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    debut = time.time()
    if suivi:
        ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        dt = time.perf_counter() - t0
        temps[j] += dt
        nb[j] += 1
        positions.append({"ply": b.ply() + 1, "joueur": j, "san": b.san(mv), "uci": mv.uci(), "secondes": round(dt, 3)})
        b.push(mv)
        positions[-1]["fen"] = b.fen()
        if suivi:
            ecrire_suivi(suivi, blancs, noirs, b, nom_ouverture, debut)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
        "positions": positions,
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def nb_gpu():
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        return max(1, sum(l.startswith("GPU") for l in r.stdout.splitlines()))
    except Exception:
        return 1


def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1, gpu_mem_mo=None):
    # plusieurs processus x plusieurs réseaux sur un GPU : sans plafond, onnxruntime réserve
    # la mémoire trop largement et le T4 sature (BFCArena / CUBLAS_STATUS_ALLOC_FAILED).
    # Les processus sont répartis sur les GPU disponibles, avec un plafond par réseau.
    global GPU_MEM_MO, GPU_ID
    n = nb_gpu()
    GPU_ID = os.getpid() % n
    GPU_MEM_MO = gpu_mem_mo or (700 if n == 1 else 1200)
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args[:5]
    suivi = args[5] if len(args) > 5 else None   # dossier de suivi en direct (facultatif)
    fichier = None
    if suivi:
        os.makedirs(suivi, exist_ok=True)
        fichier = os.path.join(suivi, f"{blancs}_{noirs}_{id_ouv}.json".replace(" ", "-").replace("@", "-"))
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies, fichier, id_ouv)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
        if fichier and os.path.exists(fichier):
            os.remove(fichier)
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
try:
    print("poids P.O.W.N. trouvés :", ce.poids("Pown/pown.onnx"))
except Exception as e:
    raise SystemExit("❌ pown.onnx introuvable : crée un Dataset Kaggle avec ce fichier et ajoute-le en Input. (" + str(e)[:100] + ")")

In [ ]:
!pip install -q onnx

In [ ]:
import os, json
SOURCES = json.loads(r'''{"types.h": "// Types de base du moteur Solution (S) : cases, pi\u00e8ces, coups, op\u00e9rations sur bitboards.\n#pragma once\n#include <cstdint>\n#include <string>\n\nusing Bitboard = uint64_t;\nusing Move = uint16_t;\n\nenum Color { WHITE = 0, BLACK = 1 };\nenum PieceType { NO_PIECE_TYPE = 0, PAWN = 1, KNIGHT = 2, BISHOP = 3, ROOK = 4, QUEEN = 5, KING = 6 };\n\n// pi\u00e8ce = couleur * 8 + type  (blancs 1..6, noirs 9..14), 0 = case vide\nconstexpr int NO_PIECE = 0;\nconstexpr int make_piece(int c, int t) { return c * 8 + t; }\nconstexpr int color_of(int p) { return p >> 3; }\nconstexpr int type_of(int p) { return p & 7; }\n\n// cases : a1 = 0, b1 = 1, ..., h8 = 63 (comme python-chess)\nconstexpr int file_of(int s) { return s & 7; }\nconstexpr int rank_of(int s) { return s >> 3; }\nconstexpr Bitboard bb(int s) { return 1ULL << s; }\n\n// coup : de (6 bits) | vers (6 bits) | promotion (type de pi\u00e8ce, 3 bits)\nconstexpr Move MOVE_NONE = 0;\nconstexpr Move make_move(int from, int to, int promo = 0) { return Move(from | (to << 6) | (promo << 12)); }\nconstexpr int from_sq(Move m) { return m & 63; }\nconstexpr int to_sq(Move m) { return (m >> 6) & 63; }\nconstexpr int promo_of(Move m) { return (m >> 12) & 7; }\n\n#if defined(_MSC_VER)\n#include <intrin.h>\ninline int lsb(Bitboard b) { unsigned long i; _BitScanForward64(&i, b); return (int)i; }\ninline int msb(Bitboard b) { unsigned long i; _BitScanReverse64(&i, b); return (int)i; }\ninline int popcount(Bitboard b) { return (int)__popcnt64(b); }\n#else\ninline int lsb(Bitboard b) { return __builtin_ctzll(b); }\ninline int msb(Bitboard b) { return 63 - __builtin_clzll(b); }\ninline int popcount(Bitboard b) { return __builtin_popcountll(b); }\n#endif\ninline int pop_lsb(Bitboard& b) { int s = lsb(b); b &= b - 1; return s; }\n\nstd::string square_name(int s);\nstd::string move_to_uci(Move m);\n", "bitboard.h": "// Tables d'attaques pr\u00e9calcul\u00e9es et attaques des pi\u00e8ces glissantes (par rayons).\n#pragma once\n#include \"types.h\"\n\nextern Bitboard KNIGHT_ATT[64], KING_ATT[64], PAWN_ATT[2][64];\nextern Bitboard RAY[8][64];          // 8 directions : N NE E NW | S SW W SE (les 4 premi\u00e8res vont vers les grands indices)\nextern Bitboard BETWEEN[64][64];     // cases strictement entre deux cases align\u00e9es (sinon 0)\nextern Bitboard LINE[64][64];        // toute la ligne passant par deux cases align\u00e9es (sinon 0)\n\nvoid init_bitboards();\n\ninline Bitboard ray_attack(int dir, int sq, Bitboard occ) {\n    Bitboard a = RAY[dir][sq];\n    Bitboard blockers = a & occ;\n    if (blockers) {\n        int s = dir < 4 ? lsb(blockers) : msb(blockers);\n        a ^= RAY[dir][s];\n    }\n    return a;\n}\ninline Bitboard rook_attacks(int sq, Bitboard occ) {\n    return ray_attack(0, sq, occ) | ray_attack(2, sq, occ) | ray_attack(4, sq, occ) | ray_attack(6, sq, occ);\n}\ninline Bitboard bishop_attacks(int sq, Bitboard occ) {\n    return ray_attack(1, sq, occ) | ray_attack(3, sq, occ) | ray_attack(5, sq, occ) | ray_attack(7, sq, occ);\n}\ninline Bitboard queen_attacks(int sq, Bitboard occ) { return rook_attacks(sq, occ) | bishop_attacks(sq, occ); }\n", "bitboard.cpp": "#include \"bitboard.h\"\n\nBitboard KNIGHT_ATT[64], KING_ATT[64], PAWN_ATT[2][64], RAY[8][64], BETWEEN[64][64], LINE[64][64];\n\nstd::string square_name(int s) { return std::string(1, char('a' + file_of(s))) + char('1' + rank_of(s)); }\n\nstd::string move_to_uci(Move m) {\n    if (m == MOVE_NONE) return \"0000\";\n    std::string s = square_name(from_sq(m)) + square_name(to_sq(m));\n    static const char pc[] = \" pnbrqk\";\n    if (promo_of(m)) s += pc[promo_of(m)];\n    return s;\n}\n\nvoid init_bitboards() {\n    static const int DR[8] = {1, 1, 0, 1, -1, -1, 0, -1};\n    static const int DF[8] = {0, 1, 1, -1, 0, -1, -1, 1};\n    for (int s = 0; s < 64; s++) {\n        int r = rank_of(s), f = file_of(s);\n        for (int d = 0; d < 8; d++) {\n            Bitboard b = 0;\n            for (int rr = r + DR[d], ff = f + DF[d]; rr >= 0 && rr < 8 && ff >= 0 && ff < 8; rr += DR[d], ff += DF[d])\n                b |= bb(rr * 8 + ff);\n            RAY[d][s] = b;\n        }\n        KNIGHT_ATT[s] = KING_ATT[s] = PAWN_ATT[0][s] = PAWN_ATT[1][s] = 0;\n        static const int KN[8][2] = {{1, 2}, {2, 1}, {2, -1}, {1, -2}, {-1, -2}, {-2, -1}, {-2, 1}, {-1, 2}};\n        for (auto& k : KN) {\n            int rr = r + k[0], ff = f + k[1];\n            if (rr >= 0 && rr < 8 && ff >= 0 && ff < 8) KNIGHT_ATT[s] |= bb(rr * 8 + ff);\n        }\n        for (int d = 0; d < 8; d++) {\n            int rr = r + DR[d], ff = f + DF[d];\n            if (rr >= 0 && rr < 8 && ff >= 0 && ff < 8) KING_ATT[s] |= bb(rr * 8 + ff);\n        }\n        for (int df : {-1, 1}) {\n            if (f + df >= 0 && f + df < 8) {\n                if (r + 1 < 8) PAWN_ATT[WHITE][s] |= bb((r + 1) * 8 + f + df);\n                if (r - 1 >= 0) PAWN_ATT[BLACK][s] |= bb((r - 1) * 8 + f + df);\n            }\n        }\n    }\n    for (int a = 0; a < 64; a++)\n        for (int b = 0; b < 64; b++) {\n            BETWEEN[a][b] = LINE[a][b] = 0;\n            for (int d = 0; d < 8; d++)\n                if (RAY[d][a] & bb(b)) {\n                    BETWEEN[a][b] = RAY[d][a] & ~RAY[d][b] & ~bb(b);\n                    LINE[a][b] = RAY[d][a] | RAY[(d + 4) % 8][a] | bb(a);\n                }\n        }\n}\n", "position.h": "// Position d'\u00e9checs : bitboards + tableau de cases, coups jouables/annulables, hachage de Zobrist.\n#pragma once\n#include <string>\n#include <vector>\n#include \"bitboard.h\"\n\nconstexpr int CASTLE_WK = 1, CASTLE_WQ = 2, CASTLE_BK = 4, CASTLE_BQ = 8;\n\n// changement de pi\u00e8ce provoqu\u00e9 par un coup (sert \u00e0 mettre \u00e0 jour le r\u00e9seau NNUE sans tout recalculer)\nstruct PieceDelta { int piece, from, to; };   // from = -1 : pi\u00e8ce ajout\u00e9e ; to = -1 : pi\u00e8ce retir\u00e9e\n\nstruct StateInfo {\n    uint64_t key;\n    int castling, ep, halfmove, captured, cap_sq;\n    Move move;\n    int dn;\n    PieceDelta d[3];\n};\n\nclass Position {\npublic:\n    Bitboard byType[7] = {}, byColor[2] = {};\n    int board[64] = {};\n    int stm = WHITE, castling = 0, ep = -1, halfmove = 0, fullmove = 1;\n    uint64_t key = 0;\n    std::vector<StateInfo> hist;\n\n    static void init();                  // tables de Zobrist\n    bool set_fen(const std::string& fen);\n    std::string fen() const;\n\n    Bitboard pieces() const { return byColor[WHITE] | byColor[BLACK]; }\n    Bitboard pieces(int c, int t) const { return byColor[c] & byType[t]; }\n    int king_sq(int c) const { return lsb(byType[KING] & byColor[c]); }\n\n    Bitboard attackers_to(int sq, Bitboard occ) const;\n    bool attacked(int sq, int by) const;\n    bool in_check() const { return attacked(king_sq(stm), stm ^ 1); }\n\n    bool do_move(Move m);                // rend false (et annule) si le coup laisse le roi en \u00e9chec\n    void undo_move();\n    void do_null();\n    void undo_null();\n\n    bool is_capture(Move m) const;\n    bool has_non_pawn_material(int c) const;\n    bool insufficient_material() const;\n    bool is_repetition(int ply_from_root) const;\n    int see(Move m) const;               // gain mat\u00e9riel estim\u00e9 de l'\u00e9change sur la case d'arriv\u00e9e (en centipions)\n    const StateInfo& last() const { return hist.back(); }\n\nprivate:\n    uint64_t ep_key() const;\n    void put(int p, int s);\n    void remove(int s);\n    void move_piece(int a, int b);\n};\n", "position.cpp": "#include \"position.h\"\n#include <sstream>\n#include <cstdlib>\n\nstatic uint64_t Z_PIECE[16][64], Z_CASTLE[16], Z_EP[8], Z_SIDE;\nstatic int CASTLE_MASK[64];\nstatic const int SEE_VAL[7] = {0, 100, 320, 330, 500, 900, 20000};\n\nstatic uint64_t rng_state = 0x9E3779B97F4A7C15ULL;\nstatic uint64_t rnd() {\n    rng_state ^= rng_state >> 12; rng_state ^= rng_state << 25; rng_state ^= rng_state >> 27;\n    return rng_state * 2685821657736338717ULL;\n}\n\nvoid Position::init() {\n    init_bitboards();\n    for (auto& a : Z_PIECE) for (auto& k : a) k = rnd();\n    for (auto& k : Z_CASTLE) k = rnd();\n    for (auto& k : Z_EP) k = rnd();\n    Z_SIDE = rnd();\n    for (int s = 0; s < 64; s++) CASTLE_MASK[s] = 15;\n    CASTLE_MASK[4] = 15 & ~(CASTLE_WK | CASTLE_WQ);\n    CASTLE_MASK[0] = 15 & ~CASTLE_WQ;\n    CASTLE_MASK[7] = 15 & ~CASTLE_WK;\n    CASTLE_MASK[60] = 15 & ~(CASTLE_BK | CASTLE_BQ);\n    CASTLE_MASK[56] = 15 & ~CASTLE_BQ;\n    CASTLE_MASK[63] = 15 & ~CASTLE_BK;\n}\n\nuint64_t Position::ep_key() const {\n    if (ep < 0) return 0;\n    if (PAWN_ATT[stm ^ 1][ep] & pieces(stm, PAWN)) return Z_EP[file_of(ep)];\n    return 0;\n}\n\nvoid Position::put(int p, int s) {\n    board[s] = p;\n    byType[type_of(p)] |= bb(s);\n    byColor[color_of(p)] |= bb(s);\n    key ^= Z_PIECE[p][s];\n}\nvoid Position::remove(int s) {\n    int p = board[s];\n    board[s] = NO_PIECE;\n    byType[type_of(p)] &= ~bb(s);\n    byColor[color_of(p)] &= ~bb(s);\n    key ^= Z_PIECE[p][s];\n}\nvoid Position::move_piece(int a, int b) {\n    int p = board[a];\n    remove(a);\n    put(p, b);\n}\n\nbool Position::set_fen(const std::string& fen) {\n    *this = Position();\n    std::istringstream ss(fen);\n    std::string placement, side, cast, epstr;\n    int hm = 0, fm = 1;\n    if (!(ss >> placement >> side)) return false;\n    if (!(ss >> cast)) cast = \"-\";\n    if (!(ss >> epstr)) epstr = \"-\";\n    ss >> hm >> fm;\n    int r = 7, f = 0;\n    for (char ch : placement) {\n        if (ch == '/') { r--; f = 0; }\n        else if (ch >= '1' && ch <= '8') f += ch - '0';\n        else {\n            const char* names = \"pnbrqk\";\n            const char* p = nullptr;\n            for (const char* q = names; *q; q++) if (*q == (ch | 32)) p = q;\n            if (!p || r < 0 || f > 7) return false;\n            int color = (ch >= 'a') ? BLACK : WHITE;\n            put(make_piece(color, int(p - names) + 1), r * 8 + f);\n            f++;\n        }\n    }\n    if (!(byType[KING] & byColor[WHITE]) || !(byType[KING] & byColor[BLACK])) return false;\n    stm = side == \"b\" ? BLACK : WHITE;\n    for (char ch : cast) {\n        if (ch == 'K') castling |= CASTLE_WK;\n        if (ch == 'Q') castling |= CASTLE_WQ;\n        if (ch == 'k') castling |= CASTLE_BK;\n        if (ch == 'q') castling |= CASTLE_BQ;\n    }\n    ep = (epstr.size() == 2) ? (epstr[1] - '1') * 8 + (epstr[0] - 'a') : -1;\n    halfmove = hm;\n    fullmove = fm;\n    if (stm == BLACK) key ^= Z_SIDE;\n    key ^= Z_CASTLE[castling] ^ ep_key();\n    return true;\n}\n\nstd::string Position::fen() const {\n    std::string s;\n    for (int r = 7; r >= 0; r--) {\n        int empty = 0;\n        for (int f = 0; f < 8; f++) {\n            int p = board[r * 8 + f];\n            if (!p) { empty++; continue; }\n            if (empty) { s += char('0' + empty); empty = 0; }\n            char c = \" pnbrqk\"[type_of(p)];\n            s += color_of(p) == WHITE ? char(c - 32) : c;\n        }\n        if (empty) s += char('0' + empty);\n        if (r) s += '/';\n    }\n    s += stm == WHITE ? \" w \" : \" b \";\n    std::string c;\n    if (castling & CASTLE_WK) c += 'K';\n    if (castling & CASTLE_WQ) c += 'Q';\n    if (castling & CASTLE_BK) c += 'k';\n    if (castling & CASTLE_BQ) c += 'q';\n    s += (c.empty() ? \"-\" : c) + \" \" + (ep >= 0 ? square_name(ep) : \"-\") + \" \" + std::to_string(halfmove) + \" \" + std::to_string(fullmove);\n    return s;\n}\n\nBitboard Position::attackers_to(int sq, Bitboard occ) const {\n    return (PAWN_ATT[BLACK][sq] & pieces(WHITE, PAWN)) | (PAWN_ATT[WHITE][sq] & pieces(BLACK, PAWN))\n         | (KNIGHT_ATT[sq] & byType[KNIGHT]) | (KING_ATT[sq] & byType[KING])\n         | (bishop_attacks(sq, occ) & (byType[BISHOP] | byType[QUEEN]))\n         | (rook_attacks(sq, occ) & (byType[ROOK] | byType[QUEEN]));\n}\n\nbool Position::attacked(int sq, int by) const {\n    Bitboard occ = pieces();\n    return (PAWN_ATT[by ^ 1][sq] & pieces(by, PAWN)) || (KNIGHT_ATT[sq] & pieces(by, KNIGHT))\n        || (KING_ATT[sq] & pieces(by, KING))\n        || (bishop_attacks(sq, occ) & (pieces(by, BISHOP) | pieces(by, QUEEN)))\n        || (rook_attacks(sq, occ) & (pieces(by, ROOK) | pieces(by, QUEEN)));\n}\n\nbool Position::is_capture(Move m) const {\n    int from = from_sq(m), to = to_sq(m);\n    if (board[to]) return true;\n    return type_of(board[from]) == PAWN && to == ep && file_of(from) != file_of(to);\n}\n\nbool Position::do_move(Move m) {\n    int us = stm, them = us ^ 1, from = from_sq(m), to = to_sq(m), promo = promo_of(m);\n    int pc = board[from], pt = type_of(pc), cap = board[to];\n    StateInfo st;\n    st.key = key; st.castling = castling; st.ep = ep; st.halfmove = halfmove; st.captured = NO_PIECE; st.cap_sq = -1;\n    st.move = m; st.dn = 0;\n    key ^= ep_key() ^ Z_CASTLE[castling];\n    int cap_sq = to;\n    if (pt == PAWN && to == ep && cap == NO_PIECE && file_of(from) != file_of(to)) {\n        cap_sq = to + (us == WHITE ? -8 : 8);\n        cap = board[cap_sq];\n    }\n    if (cap) {\n        st.captured = cap; st.cap_sq = cap_sq;\n        remove(cap_sq);\n        st.d[st.dn++] = {cap, cap_sq, -1};\n    }\n    if (promo) {\n        remove(from);\n        int np = make_piece(us, promo);\n        put(np, to);\n        st.d[st.dn++] = {pc, from, -1};\n        st.d[st.dn++] = {np, -1, to};\n    } else {\n        move_piece(from, to);\n        st.d[st.dn++] = {pc, from, to};\n    }\n    if (pt == KING && (to - from == 2 || from - to == 2)) {\n        int rf = to > from ? from + 3 : from - 4, rt = to > from ? from + 1 : from - 1;\n        int rook = board[rf];\n        move_piece(rf, rt);\n        st.d[st.dn++] = {rook, rf, rt};\n    }\n    castling &= CASTLE_MASK[from] & CASTLE_MASK[to];\n    ep = (pt == PAWN && (to - from == 16 || from - to == 16)) ? (from + to) / 2 : -1;\n    halfmove = (pt == PAWN || st.captured) ? 0 : halfmove + 1;\n    if (us == BLACK) fullmove++;\n    stm = them;\n    key ^= Z_SIDE ^ Z_CASTLE[castling] ^ ep_key();\n    hist.push_back(st);\n    if (attacked(king_sq(us), them)) {\n        undo_move();\n        return false;\n    }\n    return true;\n}\n\nvoid Position::undo_move() {\n    StateInfo st = hist.back();\n    hist.pop_back();\n    stm ^= 1;\n    int us = stm, from = from_sq(st.move), to = to_sq(st.move), promo = promo_of(st.move);\n    if (promo) {\n        remove(to);\n        put(make_piece(us, PAWN), from);\n    } else {\n        move_piece(to, from);\n    }\n    int pt = type_of(board[from]);\n    if (pt == KING && (to - from == 2 || from - to == 2)) {\n        int rf = to > from ? from + 3 : from - 4, rt = to > from ? from + 1 : from - 1;\n        move_piece(rt, rf);\n    }\n    if (st.captured) put(st.captured, st.cap_sq);\n    castling = st.castling; ep = st.ep; halfmove = st.halfmove; key = st.key;\n    if (us == BLACK) fullmove--;\n}\n\nvoid Position::do_null() {\n    StateInfo st;\n    st.key = key; st.castling = castling; st.ep = ep; st.halfmove = halfmove; st.captured = NO_PIECE; st.cap_sq = -1;\n    st.move = MOVE_NONE; st.dn = 0;\n    key ^= ep_key();\n    ep = -1;\n    stm ^= 1;\n    key ^= Z_SIDE;\n    halfmove++;\n    hist.push_back(st);\n}\n\nvoid Position::undo_null() {\n    StateInfo st = hist.back();\n    hist.pop_back();\n    stm ^= 1;\n    ep = st.ep; halfmove = st.halfmove; key = st.key;\n}\n\nbool Position::has_non_pawn_material(int c) const {\n    return (byColor[c] & (byType[KNIGHT] | byType[BISHOP] | byType[ROOK] | byType[QUEEN])) != 0;\n}\n\nbool Position::insufficient_material() const {\n    if (byType[PAWN] | byType[ROOK] | byType[QUEEN]) return false;\n    return popcount(byType[KNIGHT] | byType[BISHOP]) <= 1;\n}\n\nbool Position::is_repetition(int ply_from_root) const {\n    int n = (int)hist.size();\n    int lim = std::min(halfmove, n);\n    int count = 0;\n    for (int i = 2; i <= lim; i += 2) {\n        if (hist[n - i].key == key) {\n            if (i <= ply_from_root) return true;\n            if (++count >= 2) return true;\n        }\n    }\n    return false;\n}\n\nint Position::see(Move m) const {\n    int from = from_sq(m), to = to_sq(m);\n    int gain[32], d = 0;\n    int target = board[to] ? type_of(board[to]) : PAWN;   // en passant : un pion\n    int attacker = type_of(board[from]);\n    gain[0] = SEE_VAL[target];\n    if (promo_of(m)) gain[0] += SEE_VAL[promo_of(m)] - SEE_VAL[PAWN];\n    Bitboard occ = pieces() ^ bb(from);\n    int side = stm ^ 1;\n    int cur_val = promo_of(m) ? SEE_VAL[promo_of(m)] : SEE_VAL[attacker];\n    while (true) {\n        Bitboard att = attackers_to(to, occ) & occ & byColor[side];\n        if (!att) break;\n        int best = -1;\n        for (int t = PAWN; t <= KING; t++) {\n            Bitboard b = att & byType[t];\n            if (b) { best = lsb(b); break; }\n        }\n        d++;\n        gain[d] = cur_val - gain[d - 1];\n        if (std::max(-gain[d - 1], gain[d]) < 0) break;\n        cur_val = SEE_VAL[type_of(board[best])];\n        occ ^= bb(best);\n        side ^= 1;\n        if (d >= 30) break;\n    }\n    for (; d > 0; d--) gain[d - 1] = -std::max(-gain[d - 1], gain[d]);\n    return gain[0];\n}\n", "movegen.h": "// G\u00e9n\u00e9ration des coups pseudo-l\u00e9gaux (la l\u00e9galit\u00e9 \u2014 roi pas en \u00e9chec \u2014 est v\u00e9rifi\u00e9e par Position::do_move).\n#pragma once\n#include \"position.h\"\n\nenum GenType { GEN_ALL, GEN_CAPTURES };   // CAPTURES : prises + promotions en dame (pour la quiescence)\n\nstruct MoveList {\n    Move m[256];\n    int n = 0;\n    void add(Move mv) { m[n++] = mv; }\n};\n\nvoid generate(const Position& pos, MoveList& list, GenType type);\nMove parse_uci_move(Position& pos, const std::string& s);   // MOVE_NONE si ill\u00e9gal\nuint64_t perft(Position& pos, int depth);\n", "movegen.cpp": "#include \"movegen.h\"\n\nstatic void add_pawn_move(MoveList& l, int from, int to, bool promo_rank, GenType type) {\n    if (promo_rank) {\n        l.add(make_move(from, to, QUEEN));\n        if (type == GEN_ALL) {\n            l.add(make_move(from, to, ROOK));\n            l.add(make_move(from, to, BISHOP));\n            l.add(make_move(from, to, KNIGHT));\n        }\n    } else {\n        l.add(make_move(from, to));\n    }\n}\n\nvoid generate(const Position& pos, MoveList& list, GenType type) {\n    const int us = pos.stm, them = us ^ 1;\n    const Bitboard occ = pos.pieces(), mine = pos.byColor[us], theirs = pos.byColor[them];\n    const Bitboard target = type == GEN_ALL ? ~mine : theirs;\n\n    // pions\n    Bitboard pawns = pos.pieces(us, PAWN);\n    const int up = us == WHITE ? 8 : -8, start_rank = us == WHITE ? 1 : 6, last_rank = us == WHITE ? 6 : 1;\n    while (pawns) {\n        int from = pop_lsb(pawns);\n        bool promo = rank_of(from) == last_rank;\n        int one = from + up;\n        if (!(occ & bb(one))) {\n            if (type == GEN_ALL || promo) add_pawn_move(list, from, one, promo, type);\n            if (type == GEN_ALL && rank_of(from) == start_rank && !(occ & bb(one + up)))\n                list.add(make_move(from, one + up));\n        }\n        Bitboard caps = PAWN_ATT[us][from] & theirs;\n        while (caps) add_pawn_move(list, from, pop_lsb(caps), promo, type);\n        if (pos.ep >= 0 && (PAWN_ATT[us][from] & bb(pos.ep))) list.add(make_move(from, pos.ep));\n    }\n    // cavaliers, fous, tours, dames, roi\n    Bitboard b = pos.pieces(us, KNIGHT);\n    while (b) {\n        int from = pop_lsb(b);\n        Bitboard t = KNIGHT_ATT[from] & target;\n        while (t) list.add(make_move(from, pop_lsb(t)));\n    }\n    b = pos.pieces(us, BISHOP);\n    while (b) {\n        int from = pop_lsb(b);\n        Bitboard t = bishop_attacks(from, occ) & target;\n        while (t) list.add(make_move(from, pop_lsb(t)));\n    }\n    b = pos.pieces(us, ROOK);\n    while (b) {\n        int from = pop_lsb(b);\n        Bitboard t = rook_attacks(from, occ) & target;\n        while (t) list.add(make_move(from, pop_lsb(t)));\n    }\n    b = pos.pieces(us, QUEEN);\n    while (b) {\n        int from = pop_lsb(b);\n        Bitboard t = queen_attacks(from, occ) & target;\n        while (t) list.add(make_move(from, pop_lsb(t)));\n    }\n    int ksq = pos.king_sq(us);\n    Bitboard t = KING_ATT[ksq] & target;\n    while (t) list.add(make_move(ksq, pop_lsb(t)));\n\n    // roques (cases libres, roi pas en \u00e9chec, case travers\u00e9e non attaqu\u00e9e ; l'arriv\u00e9e est v\u00e9rifi\u00e9e par do_move)\n    if (type == GEN_ALL) {\n        if (us == WHITE && ksq == 4) {\n            if ((pos.castling & CASTLE_WK) && !(occ & (bb(5) | bb(6))) && pos.board[7] == make_piece(WHITE, ROOK)\n                && !pos.attacked(4, BLACK) && !pos.attacked(5, BLACK) && !pos.attacked(6, BLACK))\n                list.add(make_move(4, 6));\n            if ((pos.castling & CASTLE_WQ) && !(occ & (bb(1) | bb(2) | bb(3))) && pos.board[0] == make_piece(WHITE, ROOK)\n                && !pos.attacked(4, BLACK) && !pos.attacked(3, BLACK) && !pos.attacked(2, BLACK))\n                list.add(make_move(4, 2));\n        } else if (us == BLACK && ksq == 60) {\n            if ((pos.castling & CASTLE_BK) && !(occ & (bb(61) | bb(62))) && pos.board[63] == make_piece(BLACK, ROOK)\n                && !pos.attacked(60, WHITE) && !pos.attacked(61, WHITE) && !pos.attacked(62, WHITE))\n                list.add(make_move(60, 62));\n            if ((pos.castling & CASTLE_BQ) && !(occ & (bb(57) | bb(58) | bb(59))) && pos.board[56] == make_piece(BLACK, ROOK)\n                && !pos.attacked(60, WHITE) && !pos.attacked(59, WHITE) && !pos.attacked(58, WHITE))\n                list.add(make_move(60, 58));\n        }\n    }\n}\n\nMove parse_uci_move(Position& pos, const std::string& s) {\n    MoveList l;\n    generate(pos, l, GEN_ALL);\n    for (int i = 0; i < l.n; i++)\n        if (move_to_uci(l.m[i]) == s) {\n            if (pos.do_move(l.m[i])) { pos.undo_move(); return l.m[i]; }\n            return MOVE_NONE;\n        }\n    return MOVE_NONE;\n}\n\nuint64_t perft(Position& pos, int depth) {\n    MoveList l;\n    generate(pos, l, GEN_ALL);\n    uint64_t n = 0;\n    for (int i = 0; i < l.n; i++) {\n        if (!pos.do_move(l.m[i])) continue;\n        n += depth <= 1 ? 1 : perft(pos, depth - 1);\n        pos.undo_move();\n    }\n    return n;\n}\n", "nnue.h": "// R\u00e9seau P.O.W.N. (NNUE HalfKAv2_hm) en C++ : m\u00eame formule que Pown/commun_pown.py.\n//   22528 -> 256 (par point de vue, poids partag\u00e9s) -> [2x256] -> 32 -> 32 -> 1 ; activations \u00ab clipped \u00bb dans [0,1]\n//   sortie = logit ; centipions = 400 x logit (du point de vue du camp au trait)\n#pragma once\n#include <string>\n#include <vector>\n#include \"position.h\"\n\nconstexpr int NNUE_F = 22528, NNUE_H = 256, NNUE_H1 = 32, NNUE_H2 = 32;\n\nstruct Accumulator {\n    alignas(64) float v[2][NNUE_H];   // un par point de vue (blanc, noir)\n};\n\nclass NNUE {\npublic:\n    bool loaded = false;\n    bool load(const std::string& path);                          // fichier pown.bin (voir tools/exporter_poids.py)\n    void load_random(unsigned seed);                             // poids al\u00e9atoires (tests)\n    static int feature(int persp, int ksq, int piece, int sq);   // index de la caract\u00e9ristique, -1 pour son propre roi\n    void refresh(const Position& pos, Accumulator& a, int persp) const;\n    void refresh_all(const Position& pos, Accumulator& a) const { refresh(pos, a, WHITE); refresh(pos, a, BLACK); }\n    // met \u00e0 jour l'accumulateur `to` \u00e0 partir de `from` apr\u00e8s le coup dont les changements sont dans st\n    void update(const Position& pos, const Accumulator& from, Accumulator& to, const StateInfo& st) const;\n    int evaluate(const Accumulator& a, int stm) const;           // centipions, point de vue du camp au trait\nprivate:\n    std::vector<float> ft;                                       // NNUE_F x NNUE_H\n    alignas(64) float fb[NNUE_H], w1t[2 * NNUE_H * NNUE_H1], b1[NNUE_H1], w2t[NNUE_H1 * NNUE_H2], b2[NNUE_H2], w3[NNUE_H2], b3 = 0;\n};\n", "nnue.cpp": "#include \"nnue.h\"\n#include <cstdio>\n#include <cstring>\n#include <random>\n#include <algorithm>\n\nint NNUE::feature(int persp, int ksq, int piece, int sq) {\n    int pc = color_of(piece), pt = type_of(piece);\n    if (pc == persp && pt == KING) return -1;\n    int kf = file_of(ksq);\n    bool mirror = kf >= 4;\n    int bucket = rank_of(ksq) * 4 + (mirror ? 7 - kf : kf);\n    int f = file_of(sq);\n    if (mirror) f = 7 - f;\n    int t = (pc == persp) ? pt - 1 : 5 + pt - 1;\n    return bucket * 704 + (rank_of(sq) * 8 + f) * 11 + t;\n}\n\nbool NNUE::load(const std::string& path) {\n    FILE* f = fopen(path.c_str(), \"rb\");\n    if (!f) return false;\n    char magic[4];\n    int32_t version = 0, dims[4] = {0, 0, 0, 0};\n    if (fread(magic, 1, 4, f) != 4 || memcmp(magic, \"POWN\", 4) != 0 || fread(&version, 4, 1, f) != 1 || version != 1\n        || fread(dims, 4, 4, f) != 4 || dims[0] != NNUE_F || dims[1] != NNUE_H || dims[2] != NNUE_H1 || dims[3] != NNUE_H2) {\n        fclose(f);\n        return false;\n    }\n    ft.resize((size_t)NNUE_F * NNUE_H);\n    std::vector<float> w1((size_t)NNUE_H1 * 2 * NNUE_H), w2((size_t)NNUE_H2 * NNUE_H1);\n    bool ok = fread(ft.data(), 4, ft.size(), f) == ft.size() && fread(fb, 4, NNUE_H, f) == NNUE_H\n        && fread(w1.data(), 4, w1.size(), f) == w1.size() && fread(b1, 4, NNUE_H1, f) == NNUE_H1\n        && fread(w2.data(), 4, w2.size(), f) == w2.size() && fread(b2, 4, NNUE_H2, f) == NNUE_H2\n        && fread(w3, 4, NNUE_H2, f) == NNUE_H2 && fread(&b3, 4, 1, f) == 1;\n    fclose(f);\n    if (!ok) return false;\n    // torch stocke (sortie, entr\u00e9e) ; on transpose pour parcourir les entr\u00e9es en dehors, les sorties en dedans\n    for (int o = 0; o < NNUE_H1; o++)\n        for (int i = 0; i < 2 * NNUE_H; i++) w1t[i * NNUE_H1 + o] = w1[(size_t)o * 2 * NNUE_H + i];\n    for (int o = 0; o < NNUE_H2; o++)\n        for (int i = 0; i < NNUE_H1; i++) w2t[i * NNUE_H2 + o] = w2[(size_t)o * NNUE_H1 + i];\n    loaded = true;\n    return true;\n}\n\nvoid NNUE::load_random(unsigned seed) {\n    std::mt19937 g(seed);\n    std::normal_distribution<float> n01(0.f, 1.f);\n    ft.resize((size_t)NNUE_F * NNUE_H);\n    for (auto& x : ft) x = n01(g) * 0.05f;\n    for (auto& x : fb) x = 0.5f;\n    for (auto& x : w1t) x = n01(g) * 0.04f;\n    for (auto& x : b1) x = n01(g) * 0.04f;\n    for (auto& x : w2t) x = n01(g) * 0.17f;\n    for (auto& x : b2) x = n01(g) * 0.17f;\n    for (auto& x : w3) x = n01(g) * 0.17f;\n    b3 = 0.1f;\n    loaded = true;\n}\n\nvoid NNUE::refresh(const Position& pos, Accumulator& a, int persp) const {\n    float* v = a.v[persp];\n    std::memcpy(v, fb, sizeof(float) * NNUE_H);\n    int ksq = pos.king_sq(persp);\n    Bitboard occ = pos.pieces();\n    while (occ) {\n        int s = pop_lsb(occ);\n        int idx = feature(persp, ksq, pos.board[s], s);\n        if (idx < 0) continue;\n        const float* row = &ft[(size_t)idx * NNUE_H];\n        for (int j = 0; j < NNUE_H; j++) v[j] += row[j];\n    }\n}\n\nvoid NNUE::update(const Position& pos, const Accumulator& from, Accumulator& to, const StateInfo& st) const {\n    for (int persp = 0; persp < 2; persp++) {\n        // si le roi de ce point de vue a boug\u00e9, la case de r\u00e9f\u00e9rence change : on recalcule tout\n        bool king_moved = false;\n        for (int i = 0; i < st.dn; i++)\n            if (st.d[i].piece == make_piece(persp, KING) && st.d[i].from >= 0 && st.d[i].to >= 0) king_moved = true;\n        if (king_moved) { refresh(pos, to, persp); continue; }\n        float* v = to.v[persp];\n        std::memcpy(v, from.v[persp], sizeof(float) * NNUE_H);\n        int ksq = pos.king_sq(persp);\n        for (int i = 0; i < st.dn; i++) {\n            const PieceDelta& d = st.d[i];\n            if (d.from >= 0) {\n                int idx = feature(persp, ksq, d.piece, d.from);\n                if (idx >= 0) { const float* row = &ft[(size_t)idx * NNUE_H]; for (int j = 0; j < NNUE_H; j++) v[j] -= row[j]; }\n            }\n            if (d.to >= 0) {\n                int idx = feature(persp, ksq, d.piece, d.to);\n                if (idx >= 0) { const float* row = &ft[(size_t)idx * NNUE_H]; for (int j = 0; j < NNUE_H; j++) v[j] += row[j]; }\n            }\n        }\n    }\n}\n\nstatic inline float clip01(float x) { return x < 0.f ? 0.f : (x > 1.f ? 1.f : x); }\n\nint NNUE::evaluate(const Accumulator& a, int stm) const {\n    alignas(64) float x[2 * NNUE_H], h1[NNUE_H1], h2[NNUE_H2];\n    const float* us = a.v[stm];\n    const float* them = a.v[stm ^ 1];\n    for (int i = 0; i < NNUE_H; i++) { x[i] = clip01(us[i]); x[NNUE_H + i] = clip01(them[i]); }\n    std::memcpy(h1, b1, sizeof(h1));\n    for (int i = 0; i < 2 * NNUE_H; i++) {\n        float xi = x[i];\n        if (xi == 0.f) continue;                      // beaucoup de z\u00e9ros apr\u00e8s l'\u00e9cr\u00eatage : on les saute\n        const float* w = &w1t[i * NNUE_H1];\n        for (int o = 0; o < NNUE_H1; o++) h1[o] += w[o] * xi;\n    }\n    for (int o = 0; o < NNUE_H1; o++) h1[o] = clip01(h1[o]);\n    std::memcpy(h2, b2, sizeof(h2));\n    for (int i = 0; i < NNUE_H1; i++) {\n        float xi = h1[i];\n        if (xi == 0.f) continue;\n        const float* w = &w2t[i * NNUE_H2];\n        for (int o = 0; o < NNUE_H2; o++) h2[o] += w[o] * xi;\n    }\n    float out = b3;\n    for (int o = 0; o < NNUE_H2; o++) out += clip01(h2[o]) * w3[o];\n    float cp = 400.f * out;\n    return (int)std::max(-3000.f, std::min(3000.f, cp));\n}\n", "tt.h": "// Table de transposition : m\u00e9morise le r\u00e9sultat de positions d\u00e9j\u00e0 recherch\u00e9es.\n#pragma once\n#include <cstdint>\n#include <vector>\n#include \"types.h\"\n\nenum TTFlag : uint8_t { TT_NONE = 0, TT_LOWER = 1, TT_UPPER = 2, TT_EXACT = 3 };\n\nstruct TTEntry {\n    uint64_t key;\n    int16_t score;\n    Move move;\n    int8_t depth;\n    uint8_t flag;\n};\n\nclass TT {\npublic:\n    explicit TT(size_t mb = 64) { resize(mb); }\n    void resize(size_t mb) {\n        size_t n = 1;\n        while (n * 2 * sizeof(TTEntry) <= mb * 1024 * 1024) n *= 2;\n        table.assign(n, TTEntry{0, 0, 0, 0, 0});\n        mask = n - 1;\n    }\n    void clear() { std::fill(table.begin(), table.end(), TTEntry{0, 0, 0, 0, 0}); }\n    TTEntry* probe(uint64_t key) {\n        TTEntry* e = &table[key & mask];\n        return (e->key == key && e->flag != TT_NONE) ? e : nullptr;\n    }\n    void store(uint64_t key, int score, Move move, int depth, uint8_t flag) {\n        TTEntry* e = &table[key & mask];\n        // on garde l'entr\u00e9e plus profonde de la m\u00eame position ; sinon on remplace\n        if (e->key == key && e->depth > depth + 2 && flag != TT_EXACT) {\n            if (move != MOVE_NONE && e->move == MOVE_NONE) e->move = move;\n            return;\n        }\n        if (move == MOVE_NONE && e->key == key) move = e->move;\n        *e = TTEntry{key, (int16_t)score, move, (int8_t)depth, flag};\n    }\nprivate:\n    std::vector<TTEntry> table;\n    size_t mask = 0;\n};\n", "search.h": "// Recherche alpha-b\u00eata (PVS) avec approfondissement it\u00e9ratif, gestion du temps, quiescence et \u00e9lagages.\n#pragma once\n#include <atomic>\n#include <chrono>\n#include \"movegen.h\"\n#include \"nnue.h\"\n#include \"tt.h\"\n\nconstexpr int MAX_PLY = 128;\nconstexpr int INF_SCORE = 32001;\nconstexpr int MATE_SCORE = 32000;\nconstexpr int MATE_BOUND = MATE_SCORE - MAX_PLY;\n\nstruct Limits {\n    int depth = 0;\n    int64_t nodes = 0;\n    int movetime = 0, wtime = 0, btime = 0, winc = 0, binc = 0, movestogo = 0;\n    bool infinite = false;\n};\n\nstruct SearchResult {\n    Move best = MOVE_NONE;\n    int score = 0, depth = 0;\n    int64_t nodes = 0;\n};\n\nclass Search {\npublic:\n    Search(NNUE& n, TT& t) : net(n), tt(t) { init_tables(); }\n    bool use_material = false;       // \u00e9valuation \u00ab mat\u00e9riel seul \u00bb (tests de la recherche)\n    int move_overhead = 30;          // millisecondes r\u00e9serv\u00e9es par coup (communication, latence)\n    bool quiet = false;              // pas d'affichage \u00ab info \u00bb\n    SearchResult go(const Position& root, const Limits& lim, std::atomic<bool>& stop_flag);\nprivate:\n    NNUE& net;\n    TT& tt;\n    Position pos;\n    Accumulator acc[MAX_PLY + 8];\n    Move killers[MAX_PLY + 4][2];\n    int history[2][64][64];\n    Move pv[MAX_PLY + 2][MAX_PLY + 2];\n    int pv_len[MAX_PLY + 2];\n    int lmr[64][64];\n    int64_t nodes = 0, node_limit = 0;\n    std::chrono::steady_clock::time_point t0;\n    int64_t soft_ms = 0, hard_ms = 0;\n    std::atomic<bool>* stop_ptr = nullptr;\n    bool stopped = false;\n    bool ignore_time = false;        // la profondeur 1 est toujours termin\u00e9e (un coup jouable \u00e0 tout prix)\n\n    void init_tables();\n    int64_t elapsed_ms() const;\n    void check_time();\n    int eval(int ply);\n    bool make(Move m, int ply);\n    int negamax(int depth, int alpha, int beta, int ply, bool do_null);\n    int qsearch(int alpha, int beta, int ply, int qply = 0);\n    void set_time(const Limits& lim);\n    void print_info(int depth, int score, const SearchResult& r);\n};\n\nstd::string score_to_uci(int score);\n", "search.cpp": "#include \"search.h\"\n#include <algorithm>\n#include <cmath>\n#include <cstring>\n#include <iostream>\n\nstatic const int MAT_VAL[7] = {0, 100, 320, 330, 500, 900, 0};\n\nstatic inline int to_tt(int s, int ply) { return s >= MATE_BOUND ? s + ply : (s <= -MATE_BOUND ? s - ply : s); }\nstatic inline int from_tt(int s, int ply) { return s >= MATE_BOUND ? s - ply : (s <= -MATE_BOUND ? s + ply : s); }\n\nstd::string score_to_uci(int s) {\n    if (s >= MATE_BOUND) return \"mate \" + std::to_string((MATE_SCORE - s + 1) / 2);\n    if (s <= -MATE_BOUND) return \"mate \" + std::to_string(-(MATE_SCORE + s + 1) / 2);\n    return \"cp \" + std::to_string(s);\n}\n\nvoid Search::init_tables() {\n    for (int d = 1; d < 64; d++)\n        for (int m = 1; m < 64; m++) lmr[d][m] = (int)(0.75 + std::log((double)d) * std::log((double)m) / 2.25);\n    for (auto& a : lmr[0]) a = 0;\n}\n\nint64_t Search::elapsed_ms() const {\n    return std::chrono::duration_cast<std::chrono::milliseconds>(std::chrono::steady_clock::now() - t0).count();\n}\n\nvoid Search::check_time() {\n    if (stop_ptr->load(std::memory_order_relaxed)) { stopped = true; return; }\n    if (node_limit && nodes >= node_limit) { stopped = true; return; }\n    if (!ignore_time && hard_ms && elapsed_ms() >= hard_ms) stopped = true;\n}\n\nint Search::eval(int ply) {\n    if (use_material || !net.loaded) {\n        int v = 0;\n        for (int t = PAWN; t <= QUEEN; t++)\n            v += MAT_VAL[t] * (popcount(pos.pieces(pos.stm, t)) - popcount(pos.pieces(pos.stm ^ 1, t)));\n        return v;\n    }\n    return net.evaluate(acc[ply], pos.stm);\n}\n\nbool Search::make(Move m, int ply) {\n    if (!pos.do_move(m)) return false;\n    if (!use_material && net.loaded) net.update(pos, acc[ply], acc[ply + 1], pos.last());\n    return true;\n}\n\nint Search::qsearch(int alpha, int beta, int ply, int qply) {\n    nodes++;\n    if ((nodes & 1023) == 0) check_time();\n    if (stopped) return 0;\n    if (pos.is_repetition(ply) || pos.halfmove >= 100 || pos.insufficient_material()) return 0;\n    if (ply >= MAX_PLY - 1) return eval(ply);\n    bool in_check = pos.in_check();\n    int best, stand = -INF_SCORE;\n    const int orig_alpha = alpha;\n    if (TTEntry* e = tt.probe(pos.key)) {\n        int s = from_tt(e->score, ply);\n        if (e->flag == TT_EXACT || (e->flag == TT_LOWER && s >= beta) || (e->flag == TT_UPPER && s <= alpha)) return s;\n    }\n    MoveList ml;\n    if (in_check) {\n        best = -MATE_SCORE + ply;\n        generate(pos, ml, GEN_ALL);\n    } else {\n        stand = best = eval(ply);\n        if (best >= beta) return best;\n        alpha = std::max(alpha, best);\n        if (qply >= 8) return best;                                  // la quiescence ne s'\u00e9ternise pas\n        generate(pos, ml, GEN_CAPTURES);\n    }\n    int score[256];\n    for (int i = 0; i < ml.n; i++) {\n        Move m = ml.m[i];\n        int victim = pos.board[to_sq(m)] ? type_of(pos.board[to_sq(m)]) : PAWN;\n        int attacker = type_of(pos.board[from_sq(m)]);\n        score[i] = pos.is_capture(m) ? 10000 + victim * 16 - attacker : (promo_of(m) ? 9000 : 0);\n    }\n    for (int i = 0; i < ml.n; i++) {\n        int bi = i;\n        for (int j = i + 1; j < ml.n; j++) if (score[j] > score[bi]) bi = j;\n        std::swap(ml.m[i], ml.m[bi]); std::swap(score[i], score[bi]);\n        Move m = ml.m[i];\n        if (!in_check) {\n            int victim = pos.board[to_sq(m)] ? MAT_VAL[type_of(pos.board[to_sq(m)])] : 100;\n            if (!promo_of(m) && stand + victim + 200 < alpha) continue;       // m\u00eame en gagnant la pi\u00e8ce, loin de alpha\n            if (!promo_of(m) && pos.see(m) < 0) continue;                      // mauvaise capture\n        }\n        if (!make(m, ply)) continue;\n        int v = -qsearch(-beta, -alpha, ply + 1, in_check ? qply : qply + 1);\n        pos.undo_move();\n        if (stopped) return 0;\n        if (v > best) {\n            best = v;\n            if (v > alpha) { alpha = v; if (alpha >= beta) break; }\n        }\n    }\n    tt.store(pos.key, to_tt(best, ply), MOVE_NONE, 0, best >= beta ? TT_LOWER : (best > orig_alpha ? TT_EXACT : TT_UPPER));\n    return best;\n}\n\nint Search::negamax(int depth, int alpha, int beta, int ply, bool do_null) {\n    pv_len[ply] = ply;\n    if (stopped) return 0;\n    const bool pv_node = beta - alpha > 1, root = ply == 0;\n    if (!root) {\n        if (pos.is_repetition(ply) || pos.halfmove >= 100 || pos.insufficient_material()) return 0;\n        if (ply >= MAX_PLY - 1) return eval(ply);\n        alpha = std::max(alpha, -MATE_SCORE + ply);\n        beta = std::min(beta, MATE_SCORE - ply - 1);\n        if (alpha >= beta) return alpha;\n    }\n    const bool in_check = pos.in_check();\n    if (in_check) depth++;\n    if (depth <= 0) return qsearch(alpha, beta, ply);\n    nodes++;\n    if ((nodes & 1023) == 0) check_time();\n    if (stopped) return 0;\n\n    Move tt_move = MOVE_NONE;\n    if (TTEntry* e = tt.probe(pos.key)) {\n        tt_move = e->move;\n        if (!pv_node && e->depth >= depth) {\n            int s = from_tt(e->score, ply);\n            if (e->flag == TT_EXACT || (e->flag == TT_LOWER && s >= beta) || (e->flag == TT_UPPER && s <= alpha)) return s;\n        }\n    }\n    const int us = pos.stm;\n    int static_eval = in_check ? -INF_SCORE : eval(ply);\n\n    if (!pv_node && !in_check && std::abs(beta) < MATE_BOUND) {\n        // \u00e9lagage \u00ab inverse \u00bb : trop bien pour \u00eatre inqui\u00e9t\u00e9 \u00e0 faible profondeur\n        if (depth <= 6 && static_eval - 90 * depth >= beta) return static_eval;\n        // coup nul : m\u00eame en passant son tour on reste au-dessus de beta\n        if (do_null && depth >= 3 && static_eval >= beta && pos.has_non_pawn_material(us)) {\n            int R = 3 + depth / 4 + std::min((static_eval - beta) / 200, 2);\n            pos.do_null();\n            acc[ply + 1] = acc[ply];\n            int v = -negamax(depth - 1 - R, -beta, -beta + 1, ply + 1, false);\n            pos.undo_null();\n            if (stopped) return 0;\n            if (v >= beta) return v >= MATE_BOUND ? beta : v;\n        }\n    }\n\n    MoveList ml;\n    generate(pos, ml, GEN_ALL);\n    int score[256];\n    for (int i = 0; i < ml.n; i++) {\n        Move m = ml.m[i];\n        if (m == tt_move) { score[i] = 1 << 30; continue; }\n        if (pos.is_capture(m)) {\n            int victim = pos.board[to_sq(m)] ? type_of(pos.board[to_sq(m)]) : PAWN;\n            int attacker = type_of(pos.board[from_sq(m)]);\n            int mvv = victim * 16 - attacker;\n            score[i] = (attacker <= victim || pos.see(m) >= 0) ? 500000 + mvv : -100000 + mvv;\n        } else if (promo_of(m)) {\n            score[i] = promo_of(m) == QUEEN ? 600000 : -200000;\n        } else if (m == killers[ply][0]) score[i] = 400000;\n        else if (m == killers[ply][1]) score[i] = 390000;\n        else score[i] = history[us][from_sq(m)][to_sq(m)];\n    }\n\n    int best = -INF_SCORE, move_count = 0, nq = 0;\n    Move best_move = MOVE_NONE, quiets[64];\n    const int orig_alpha = alpha;\n    for (int i = 0; i < ml.n; i++) {\n        int bi = i;\n        for (int j = i + 1; j < ml.n; j++) if (score[j] > score[bi]) bi = j;\n        std::swap(ml.m[i], ml.m[bi]); std::swap(score[i], score[bi]);\n        Move m = ml.m[i];\n        const bool capture = pos.is_capture(m), is_quiet = !capture && !promo_of(m);\n        if (!pv_node && !in_check && best > -MATE_BOUND && move_count > 0) {\n            if (is_quiet && depth <= 4 && move_count >= 6 + depth * depth) continue;           // trop de coups calmes d\u00e9j\u00e0 essay\u00e9s\n            if (capture && depth <= 3 && pos.see(m) < -90 * depth) continue;                    // perd trop de mat\u00e9riel\n        }\n        if (!make(m, ply)) continue;\n        move_count++;\n        const bool gives_check = pos.in_check();\n        if (!pv_node && !in_check && is_quiet && !gives_check && best > -MATE_BOUND && depth <= 3\n            && move_count > 1 && static_eval + 100 + 120 * depth <= alpha) {                 // futilit\u00e9 : sans espoir de remonter\n            pos.undo_move();\n            continue;\n        }\n        int new_depth = depth - 1, v;\n        int r = 0;\n        if (depth >= 3 && move_count > 3 && is_quiet && !in_check) {\n            r = lmr[std::min(depth, 63)][std::min(move_count, 63)];\n            if (!pv_node) r++;\n            if (gives_check) r--;\n            if (m == killers[ply][0] || m == killers[ply][1]) r--;\n            r = std::max(0, std::min(r, new_depth - 1));\n        }\n        if (move_count == 1) {\n            v = -negamax(new_depth, -beta, -alpha, ply + 1, true);\n        } else {\n            v = -negamax(new_depth - r, -alpha - 1, -alpha, ply + 1, true);\n            if (v > alpha && r > 0) v = -negamax(new_depth, -alpha - 1, -alpha, ply + 1, true);\n            if (v > alpha && v < beta) v = -negamax(new_depth, -beta, -alpha, ply + 1, true);\n        }\n        pos.undo_move();\n        if (stopped) return 0;\n        if (v > best) {\n            best = v;\n            best_move = m;\n            if (v > alpha) {\n                alpha = v;\n                pv[ply][ply] = m;\n                for (int k = ply + 1; k < pv_len[ply + 1]; k++) pv[ply][k] = pv[ply + 1][k];\n                pv_len[ply] = pv_len[ply + 1];\n                if (alpha >= beta) {\n                    if (is_quiet) {\n                        if (killers[ply][0] != m) { killers[ply][1] = killers[ply][0]; killers[ply][0] = m; }\n                        int bonus = std::min(depth * depth, 400);\n                        auto upd = [&](Move q, int b) {\n                            int& h = history[us][from_sq(q)][to_sq(q)];\n                            h += b - h * std::abs(b) / 16384;\n                        };\n                        upd(m, bonus);\n                        for (int k = 0; k < nq; k++) upd(quiets[k], -bonus);\n                    }\n                    break;\n                }\n            }\n        }\n        if (is_quiet && nq < 64) quiets[nq++] = m;\n    }\n    if (move_count == 0) return in_check ? -MATE_SCORE + ply : 0;\n    uint8_t flag = best >= beta ? TT_LOWER : (best > orig_alpha ? TT_EXACT : TT_UPPER);\n    tt.store(pos.key, to_tt(best, ply), best_move, depth, flag);\n    return best;\n}\n\nvoid Search::set_time(const Limits& lim) {\n    soft_ms = hard_ms = 0;\n    int ov = move_overhead;\n    if (lim.movetime) {\n        soft_ms = hard_ms = std::max(5, lim.movetime - ov);\n    } else if (lim.wtime || lim.btime) {\n        int64_t t = pos.stm == WHITE ? lim.wtime : lim.btime, inc = pos.stm == WHITE ? lim.winc : lim.binc;\n        int mtg = lim.movestogo ? std::min(lim.movestogo, 40) : 30;\n        int64_t base = t / mtg + inc * 3 / 4;\n        soft_ms = std::max<int64_t>(5, std::min<int64_t>(base, t * 8 / 10) - ov);\n        hard_ms = std::max<int64_t>(soft_ms, std::min<int64_t>(base * 3, t * 4 / 10) - ov);\n    }\n}\n\nvoid Search::print_info(int depth, int score, const SearchResult& r) {\n    if (quiet) return;\n    int64_t ms = elapsed_ms();\n    std::cout << \"info depth \" << depth << \" score \" << score_to_uci(score) << \" nodes \" << nodes\n              << \" nps \" << (ms ? nodes * 1000 / ms : nodes) << \" time \" << ms << \" pv\";\n    for (int i = 0; i < pv_len[0]; i++) std::cout << \" \" << move_to_uci(pv[0][i]);\n    std::cout << std::endl;\n    (void)r;\n}\n\nSearchResult Search::go(const Position& root, const Limits& lim, std::atomic<bool>& stop_flag) {\n    pos = root;\n    stop_ptr = &stop_flag;\n    stopped = false;\n    nodes = 0;\n    node_limit = lim.nodes;\n    t0 = std::chrono::steady_clock::now();\n    set_time(lim);\n    std::memset(killers, 0, sizeof(killers));\n    std::memset(history, 0, sizeof(history));\n    if (!use_material && net.loaded) net.refresh_all(pos, acc[0]);\n\n    SearchResult res;\n    MoveList ml;\n    generate(pos, ml, GEN_ALL);\n    for (int i = 0; i < ml.n; i++)\n        if (pos.do_move(ml.m[i])) { pos.undo_move(); if (res.best == MOVE_NONE) res.best = ml.m[i]; }\n    if (res.best == MOVE_NONE) return res;           // aucun coup l\u00e9gal\n\n    const int max_depth = lim.depth ? std::min(lim.depth, MAX_PLY - 2) : MAX_PLY - 2;\n    int score = 0;\n    for (int d = 1; d <= max_depth; d++) {\n        ignore_time = (d == 1);\n        int alpha = -INF_SCORE, beta = INF_SCORE, delta = 25;\n        if (d >= 5) { alpha = std::max(score - delta, -INF_SCORE); beta = std::min(score + delta, INF_SCORE); }\n        int v;\n        while (true) {\n            v = negamax(d, alpha, beta, 0, false);\n            if (stopped) break;\n            if (v <= alpha && alpha > -INF_SCORE) { alpha = std::max(alpha - delta * 2, -INF_SCORE); delta *= 2; }\n            else if (v >= beta && beta < INF_SCORE) { beta = std::min(beta + delta * 2, INF_SCORE); delta *= 2; }\n            else break;\n        }\n        if (stopped) break;\n        score = v;\n        if (pv_len[0] > 0 && pv[0][0] != MOVE_NONE) res.best = pv[0][0];\n        res.score = score; res.depth = d; res.nodes = nodes;\n        print_info(d, score, res);\n        if (lim.infinite) continue;\n        if (soft_ms && elapsed_ms() * 2 > soft_ms) break;          // la profondeur suivante d\u00e9passerait le temps pr\u00e9vu\n        if (std::abs(score) >= MATE_BOUND && d >= 6 && !lim.depth) break;\n    }\n    res.nodes = nodes;\n    return res;\n}\n", "main.cpp": "// Solution (S) : moteur d'\u00e9checs UCI utilisant le r\u00e9seau P.O.W.N.\n//   ./solution [pown.bin]          puis commandes UCI sur l'entr\u00e9e standard\n//   ./solution perft 5             nombre de positions (test des r\u00e8gles)\n//   ./solution bench               vitesse de recherche\n#include <iostream>\n#include <sstream>\n#include <thread>\n#include <memory>\n#include <fstream>\n#include \"search.h\"\n\nstatic const char* NAME = \"Solution S 1.0 (P.O.W.N.)\";\n\nstruct Engine {\n    NNUE net;\n    TT tt{64};\n    Search search{net, tt};\n    Position pos;\n    std::atomic<bool> stop_flag{false};\n    std::thread worker;\n\n    Engine() { pos.set_fen(\"rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\"); }\n    void wait() { if (worker.joinable()) worker.join(); }\n    ~Engine() { stop_flag = true; wait(); }\n\n    void set_position(std::istringstream& in) {\n        std::string tok, fen;\n        in >> tok;\n        if (tok == \"startpos\") {\n            pos.set_fen(\"rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\");\n            in >> tok;\n        } else if (tok == \"fen\") {\n            while (in >> tok && tok != \"moves\") fen += tok + \" \";\n            pos.set_fen(fen);\n        }\n        if (tok == \"moves\") {\n            while (in >> tok) {\n                Move m = parse_uci_move(pos, tok);\n                if (m == MOVE_NONE || !pos.do_move(m)) { std::cout << \"info string coup ill\u00e9gal : \" << tok << std::endl; break; }\n            }\n        }\n    }\n\n    void go(std::istringstream& in) {\n        wait();\n        Limits lim;\n        std::string tok;\n        while (in >> tok) {\n            if (tok == \"depth\") in >> lim.depth;\n            else if (tok == \"nodes\") in >> lim.nodes;\n            else if (tok == \"movetime\") in >> lim.movetime;\n            else if (tok == \"wtime\") in >> lim.wtime;\n            else if (tok == \"btime\") in >> lim.btime;\n            else if (tok == \"winc\") in >> lim.winc;\n            else if (tok == \"binc\") in >> lim.binc;\n            else if (tok == \"movestogo\") in >> lim.movestogo;\n            else if (tok == \"infinite\") lim.infinite = true;\n            else if (tok == \"perft\") { int d; in >> d; std::cout << \"nodes \" << perft(pos, d) << std::endl; return; }\n        }\n        if (!net.loaded && !search.use_material) {\n            std::cout << \"info string ERREUR : poids P.O.W.N. non charg\u00e9s (option EvalFile ou argument pown.bin)\" << std::endl;\n        }\n        stop_flag = false;\n        Position root = pos;\n        worker = std::thread([this, root, lim]() {\n            SearchResult r = search.go(root, lim, stop_flag);\n            std::cout << \"bestmove \" << move_to_uci(r.best) << std::endl;\n        });\n    }\n};\n\nstatic void bench(Engine& e) {\n    const char* fens[] = {\n        \"rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\",\n        \"r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1\",\n        \"r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4\",\n        \"8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 w - - 0 1\",\n        \"2r2rk1/1p2qppp/p1n1pn2/3p4/3P4/P1NBPN2/1P3PPP/2RQ1RK1 w - - 0 1\",\n        \"r2q1rk1/ppp2ppp/2np1n2/2b1p1B1/2B1P1b1/2NP1N2/PPP2PPP/R2Q1RK1 w - - 0 8\",\n    };\n    int64_t total = 0;\n    auto t0 = std::chrono::steady_clock::now();\n    e.search.quiet = true;\n    for (auto f : fens) {\n        Position p; p.set_fen(f);\n        e.tt.clear();\n        Limits l; l.depth = 14; l.nodes = 300000;   // born\u00e9 : dur\u00e9e pr\u00e9visible\n        std::atomic<bool> st{false};\n        SearchResult r = e.search.go(p, l, st);\n        total += r.nodes;\n        std::cout << f << \" -> \" << move_to_uci(r.best) << \" score \" << r.score << \" noeuds \" << r.nodes << std::endl;\n    }\n    double s = std::chrono::duration<double>(std::chrono::steady_clock::now() - t0).count();\n    std::cout << \"total \" << total << \" noeuds, \" << s << \" s, \" << (int64_t)(total / s) << \" noeuds/s\" << std::endl;\n}\n\nint main(int argc, char** argv) {\n    Position::init();\n    Engine e;\n    std::string weights, action;\n    int action_arg = 0;\n    for (int i = 1; i < argc; i++) {\n        std::string a = argv[i];\n        if (a == \"--random\") e.net.load_random(1);\n        else if (a == \"--material\") e.search.use_material = true;\n        else if (a == \"perft\" && i + 1 < argc) { action = a; action_arg = std::atoi(argv[++i]); }\n        else if (a == \"bench\") action = a;\n        else weights = a;\n    }\n    if (!weights.empty() && !e.net.load(weights)) std::cout << \"info string impossible de lire \" << weights << std::endl;\n    if (action == \"perft\") {\n        Position p; p.set_fen(\"rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1\");\n        std::cout << perft(p, action_arg) << std::endl;\n        return 0;\n    }\n    if (action == \"bench\") {\n        if (!e.net.loaded && !e.search.use_material) e.net.load_random(1);\n        bench(e);\n        return 0;\n    }\n\n    std::string line;\n    while (std::getline(std::cin, line)) {\n        std::istringstream in(line);\n        std::string cmd;\n        in >> cmd;\n        if (cmd == \"uci\") {\n            std::cout << \"id name \" << NAME << \"\\nid author Liam\\n\"\n                      << \"option name Hash type spin default 64 min 1 max 4096\\n\"\n                      << \"option name EvalFile type string default pown.bin\\n\"\n                      << \"option name Threads type spin default 1 min 1 max 1\\n\"\n                      << \"option name Move Overhead type spin default 30 min 0 max 5000\\n\"\n                      << \"option name Material type check default false\\n\"\n                      << \"uciok\" << std::endl;\n        } else if (cmd == \"isready\") {\n            std::cout << \"readyok\" << std::endl;\n        } else if (cmd == \"ucinewgame\") {\n            e.wait(); e.tt.clear();\n        } else if (cmd == \"setoption\") {\n            std::string tok, name, value;\n            in >> tok;\n            while (in >> tok && tok != \"value\") name += (name.empty() ? \"\" : \" \") + tok;\n            std::getline(in, value);\n            if (!value.empty() && value[0] == ' ') value.erase(0, 1);\n            if (name == \"Hash\") { e.wait(); e.tt.resize(std::max(1, std::atoi(value.c_str()))); }\n            else if (name == \"EvalFile\") { if (!e.net.load(value)) std::cout << \"info string impossible de lire \" << value << std::endl; }\n            else if (name == \"Move Overhead\") e.search.move_overhead = std::atoi(value.c_str());\n            else if (name == \"Material\") e.search.use_material = (value == \"true\");\n        } else if (cmd == \"position\") {\n            e.wait(); e.set_position(in);\n        } else if (cmd == \"go\") {\n            e.go(in);\n        } else if (cmd == \"stop\") {\n            e.stop_flag = true; e.wait();\n        } else if (cmd == \"d\") {\n            std::cout << e.pos.fen() << std::endl;\n        } else if (cmd == \"eval\") {\n            Accumulator a; e.net.refresh_all(e.pos, a);\n            std::cout << \"eval \" << e.net.evaluate(a, e.pos.stm) << \" cp (point de vue du camp au trait)\" << std::endl;\n        } else if (cmd == \"legalmoves\") {\n            MoveList l; generate(e.pos, l, GEN_ALL);\n            for (int i = 0; i < l.n; i++) if (e.pos.do_move(l.m[i])) { e.pos.undo_move(); std::cout << move_to_uci(l.m[i]) << \" \"; }\n            std::cout << std::endl;\n        } else if (cmd == \"bench\") {\n            e.wait(); if (!e.net.loaded && !e.search.use_material) e.net.load_random(1); bench(e);\n        } else if (cmd == \"quit\") {\n            break;\n        }\n    }\n    e.stop_flag = true; e.wait();\n    return 0;\n}\n", "exporter_poids.py": "\"\"\"Exporte les poids de P.O.W.N. au format binaire lu par le moteur C++ (pown.bin).\n\n    python exporter_poids.py pown.onnx pown.bin     (n\u00e9cessite le paquet onnx)\n    python exporter_poids.py pown.pt   pown.bin     (n\u00e9cessite torch)\n\nFormat (little-endian) : \"POWN\", version=1, dims = [22528, 256, 32, 32], puis en float32 :\ntransformateur (22528x256), biais (256), c1.weight (32x512), c1.bias, c2.weight (32x32), c2.bias, sortie.weight (1x32), sortie.bias\n\"\"\"\nimport struct\nimport sys\n\nimport numpy as np\n\nORDRE = [\"transformateur.weight\", \"biais\", \"c1.weight\", \"c1.bias\", \"c2.weight\", \"c2.bias\", \"sortie.weight\", \"sortie.bias\"]\nFORMES = [(22528, 256), (256,), (32, 512), (32,), (32, 32), (32,), (1, 32), (1,)]\n\n\ndef ecrire_bin(poids, chemin_bin):\n    with open(chemin_bin, \"wb\") as f:\n        f.write(b\"POWN\")\n        f.write(struct.pack(\"<i\", 1))\n        f.write(struct.pack(\"<4i\", 22528, 256, 32, 32))\n        for nom, forme in zip(ORDRE, FORMES):\n            a = np.asarray(poids[nom], dtype=np.float32)\n            assert a.shape == forme, (nom, a.shape, forme)\n            f.write(np.ascontiguousarray(a).tobytes())\n    print(\"\u00e9crit\", chemin_bin)\n\n\ndef poids_depuis_modele(modele):\n    return {k: v.detach().cpu().numpy() for k, v in modele.state_dict().items()}\n\n\ndef poids_depuis_onnx(chemin):\n    import onnx\n    from onnx import numpy_helper\n    return {i.name: numpy_helper.to_array(i) for i in onnx.load(chemin).graph.initializer}\n\n\ndef exporter(modele, chemin_bin):\n    ecrire_bin(poids_depuis_modele(modele), chemin_bin)\n\n\nif __name__ == \"__main__\":\n    src, dest = sys.argv[1], sys.argv[2]\n    if src.endswith(\".onnx\"):\n        ecrire_bin(poids_depuis_onnx(src), dest)\n    else:\n        import os\n        import torch\n        sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), \"..\", \"..\", \"Pown\"))\n        import commun_pown as pown\n        m = pown.ReseauPOWN()\n        pt = torch.load(src, map_location=\"cpu\")\n        m.load_state_dict(pt[\"modele\"] if \"modele\" in pt else pt)\n        exporter(m, dest)\n"}''')

os.makedirs("solution/src", exist_ok=True)
os.makedirs("solution/tools", exist_ok=True)
for nom, texte in SOURCES.items():
    dossier = "solution/tools" if nom.endswith(".py") else "solution/src"
    with open(os.path.join(dossier, nom), "w", encoding="utf-8") as f:
        f.write(texte)
print(len(SOURCES), "fichiers écrits dans solution/")

In [ ]:
import subprocess
cpp = [f"solution/src/{n}" for n in SOURCES if n.endswith(".cpp")]
cmd = ["g++", "-O3", "-march=native", "-std=c++17", "-pthread", "-o", "solution/solution"] + cpp
t = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stderr[-3000:] if r.returncode else f"compilé en {time.time() - t:.0f} s")
assert r.returncode == 0, "échec de la compilation"

In [ ]:
sys.path.insert(0, "solution/tools")
import exporter_poids as ex
src = ce.poids("Pown/pown.onnx")
ex.ecrire_bin(ex.poids_depuis_onnx(src), "solution/pown.bin")
os.environ["SOLUTION_BIN"] = os.path.abspath("solution/solution")
os.environ["SOLUTION_POIDS"] = os.path.abspath("solution/pown.bin")

In [ ]:
from commun_echecs import features_pown as pown_features

exe, poids = os.environ["SOLUTION_BIN"], os.environ["SOLUTION_POIDS"]
# 1) règles : nombre de positions atteignables depuis le départ (valeur officielle à profondeur 5 : 4 865 609)
n = int(subprocess.run([exe, poids, "perft", "5"], capture_output=True, text=True).stdout.split()[-1])
print("perft(5) =", f"{n:,}", "✅" if n == 4865609 else "❌ ATTENDU 4,865,609")
assert n == 4865609

# 2) le C++ doit donner la même évaluation que onnxruntime (même réseau)
fens = [chess.STARTING_FEN, "r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1",
        "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1", "8/2p5/3p4/KP5r/1R3p1k/8/4P1P1/8 b - - 0 1"]
proc = subprocess.Popen([exe, poids], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
def interroger(cmd, attendre="eval"):
    proc.stdin.write(cmd + "\n"); proc.stdin.flush()
for f in fens:
    interroger("position fen " + f); interroger("eval")
    cpp = int(proc.stdout.readline().split()[1])
    b = chess.Board(f)
    sess = ort.InferenceSession(src, providers=["CPUExecutionProvider"])
    nn, ee = pown_features(b, b.turn), pown_features(b, not b.turn)
    onnx_cp = 400 * float(sess.run(None, {"idx_nous": np.array(nn, dtype=np.int64), "dec_nous": np.array([0], dtype=np.int64),
                                          "idx_eux": np.array(ee, dtype=np.int64), "dec_eux": np.array([0], dtype=np.int64)})[0][0])
    print(f"évaluation  C++ : {cpp:6d} cp | onnxruntime : {onnx_cp:8.1f} cp", "✅" if abs(cpp - onnx_cp) < 2 else "❌")
    assert abs(cpp - onnx_cp) < 2
proc.stdin.write("quit\n"); proc.stdin.flush()

# 3) vitesse et profondeur sur des positions réelles
print(subprocess.run([exe, poids, "bench"], capture_output=True, text=True).stdout)

In [ ]:
# un coup de démonstration : profondeur et vitesse de S à 1 s par coup
eng = chess.engine.SimpleEngine.popen_uci([os.environ["SOLUTION_BIN"], os.environ["SOLUTION_POIDS"]])
for fen in ["r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1", chess.STARTING_FEN]:
    info = eng.analyse(chess.Board(fen), chess.engine.Limit(time=1))
    print(f"profondeur {info.get('depth')} | {info.get('nodes', 0):,} noeuds | {info.get('nps', 0):,} noeuds/s | score {info.get('score')} | coup {info['pv'][0] if info.get('pv') else '?'}")
eng.quit()

In [ ]:
SF = ce.installer_stockfish(os.path.join(os.getcwd(), "stockfish"))
print("Stockfish :", SF)

In [ ]:
# ---- réglages ----
JOUEUR = "POWN@S1"    # moteur C++ Solution (S), 1 s par coup
NIVEAUX = [1500, 1800, 2100, 2400, 2700, 3000]   # UCI_Elo de Stockfish (de 1320 à 3190)
SF_TEMPS = 0.1          # secondes par coup pour Stockfish
N_OUVERTURES = 8        # parties par niveau = 2 x N_OUVERTURES
MAX_PLIES = 200         # au-delà : nulle (évite les finales interminables)
NB_WORKERS = os.cpu_count()
BUDGET_H = 11           # on arrête de lancer des parties après ce temps (limite Kaggle : 12 h)
SEED = 0
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

OUVERTURES = {
    "Italienne": "e2e4 e7e5 g1f3 b8c6 f1c4 f8c5",
    "Espagnole": "e2e4 e7e5 g1f3 b8c6 f1b5 a7a6",
    "Sicilienne Najdorf": "e2e4 c7c5 g1f3 d7d6 d2d4 c5d4 f3d4 g8f6 b1c3 a7a6",
    "Française": "e2e4 e7e6 d2d4 d7d5 b1c3 g8f6",
    "Caro-Kann": "e2e4 c7c6 d2d4 d7d5 e4e5 c8f5",
    "Gambit dame refusé": "d2d4 d7d5 c2c4 e7e6 b1c3 g8f6",
    "Slave": "d2d4 d7d5 c2c4 c7c6 g1f3 g8f6",
    "Est-indienne": "d2d4 g8f6 c2c4 g7g6 b1c3 f8g7 e2e4 d7d6",
    "Nimzo-indienne": "d2d4 g8f6 c2c4 e7e6 b1c3 f8b4",
    "Anglaise": "c2c4 e7e5 b1c3 g8f6 g1f3 b8c6",
    "Scandinave": "e2e4 d7d5 e4d5 d8d5 b1c3 d5a5",
    "Londres": "d2d4 d7d5 g1f3 g8f6 c1f4 c7c5",
    "Pirc": "e2e4 d7d6 d2d4 g8f6 b1c3 g7g6",
    "Hollandaise": "d2d4 f7f5 g1f3 g8f6 g2g3 e7e6",
    "Écossaise": "e2e4 e7e5 g1f3 b8c6 d2d4 e5d4 f3d4 g8f6",
    "Réti": "g1f3 d7d5 g2g3 g8f6 f1g2 e7e6",
}
for nom, coups in OUVERTURES.items():  # vérifie que les ouvertures sont légales
    b = chess.Board()
    for u in coups.split():
        assert chess.Move.from_uci(u) in b.legal_moves, (nom, u)
        b.push_uci(u)
OUV = dict(list(OUVERTURES.items())[:N_OUVERTURES])

STOCKFISH = {f"SF@elo{e}": e for e in NIVEAUX}
taches = []
for sf in STOCKFISH:
    for nom, coups in OUV.items():
        taches.append((JOUEUR, sf, coups.split(), MAX_PLIES, nom))
        taches.append((sf, JOUEUR, coups.split(), MAX_PLIES, nom))
random.Random(SEED).shuffle(taches)  # si ça coupe, chaque niveau a à peu près le même nombre de parties
print(len(STOCKFISH), "niveaux,", len(taches), "parties")

## Parties (en parallèle, avec reprise)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

PARTIES = f"{SORTIE}/solution_parties.jsonl"
DOSSIER = f"{SORTIE}/solution_parties"          # un sous-dossier par niveau, un fichier par partie
PAR_NIVEAU = 2 * len(OUV)
EMOJI = {"victoire": "✅", "nulle": "🤝", "défaite": "❌"}

def infos(r):
    pwn_blancs = r["blancs"] == JOUEUR
    adv = r["noirs"] if pwn_blancs else r["blancs"]
    couleur = "blancs" if pwn_blancs else "noirs"
    if r["resultat"] == "1/2-1/2":
        res = "nulle"
    else:
        res = "victoire" if (r["resultat"] == "1-0") == pwn_blancs else "défaite"
    return adv, couleur, res

def enregistrer(r):
    # fichiers lisibles pour chaque partie : PGN (à ouvrir sur lichess.org/paste) + CSV de toutes les positions
    adv, couleur, res = infos(r)
    d = os.path.join(DOSSIER, adv.replace("SF@", "stockfish_"))
    os.makedirs(d, exist_ok=True)
    nom = f"{r['ouverture']}_PWN-{couleur}".replace(" ", "-")
    with open(os.path.join(d, nom + ".pgn"), "w") as f:
        f.write(r["pgn"])
    pos = pd.DataFrame(r["positions"])
    pos.insert(0, "ouverture", r["ouverture"]); pos.insert(1, "PWN_joue", couleur); pos["resultat_PWN"] = res
    pos.to_csv(os.path.join(d, nom + "_positions.csv"), index=False)

def bilan_niveau(adv, lignes):
    res = [infos(r)[2] for r in lignes]
    v, n, p = res.count("victoire"), res.count("nulle"), res.count("défaite")
    print(f"\n🏁 NIVEAU {adv} TERMINÉ : {v} victoires, {n} nulles, {p} défaites → score {100 * (v + n / 2) / len(res):.0f} %")
    for r in sorted(lignes, key=lambda r: (r["ouverture"], infos(r)[1])):
        a, c, x = infos(r)
        print(f"   {EMOJI[x]} {r['ouverture']:20s} POWN avec les {c:6s} → {x:8s} ({r['fin']}, {r['plies'] // 2} coups)")
    print()

# reprise : si l'output est vide mais qu'un ancien solution_parties.jsonl a été ajouté en input
# (dataset ou output d'une version précédente), on repart de là
import glob, shutil
if not os.path.exists(PARTIES):
    anciens = sorted(glob.glob("/kaggle/input/**/solution_parties.jsonl", recursive=True))
    if anciens:
        with open(PARTIES, "w") as f:
            for a in anciens:
                f.write(open(a).read().rstrip("\n") + "\n")
        print("reprise depuis :", *anciens)
deja = []
if os.path.exists(PARTIES):
    vus = set()
    for l in open(PARTIES):
        if l.strip():
            r = json.loads(l)
            if (r["blancs"], r["noirs"], r["ouverture"]) not in vus:
                vus.add((r["blancs"], r["noirs"], r["ouverture"]))
                deja.append(r)
    for r in deja:   # recrée les fichiers PGN / positions des parties déjà jouées
        if "positions" in r:
            enregistrer(r)
fait = {(r["blancs"], r["noirs"], r["ouverture"]) for r in deja}
par_niveau = {sf: [r for r in deja if infos(r)[0] == sf] for sf in STOCKFISH}
reste = [t for t in taches if (t[0], t[1], t[4]) not in fait]
print(len(fait), "déjà jouées,", len(reste), "à jouer")
for sf, lignes in par_niveau.items():
    if len(lignes) >= PAR_NIVEAU:
        bilan_niveau(sf, lignes)

# ---- affichage en direct : une carte par partie en cours + tableau des scores ----
import glob, shutil
import chess.svg
from concurrent.futures import wait, FIRST_COMPLETED
from IPython.display import display, HTML

SUIVI = "/tmp/solution_en_cours"
shutil.rmtree(SUIVI, ignore_errors=True); os.makedirs(SUIVI)
derniers = []     # parties finies (HTML), les plus récentes à la fin
vu = {}           # partie -> dernier demi-coup affiché (pour n'animer que les nouveaux coups)
VALEUR = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
SYMB = {chess.WHITE: {chess.PAWN: "♙", chess.KNIGHT: "♘", chess.BISHOP: "♗", chess.ROOK: "♖", chess.QUEEN: "♕"},
        chess.BLACK: {chess.PAWN: "♟", chess.KNIGHT: "♞", chess.BISHOP: "♝", chess.ROOK: "♜", chess.QUEEN: "♛"}}
DEPART = {chess.PAWN: 8, chess.KNIGHT: 2, chess.BISHOP: 2, chess.ROOK: 2, chess.QUEEN: 1}
COULEURS = {"square light": "#eeeed2", "square dark": "#769656",
            "square light lastmove": "#f6f669", "square dark lastmove": "#baca2b", "margin": "#312e2b", "coord": "#ddd"}
TAILLE = 45   # taille d'une case dans le repère SVG de chess.svg (marge 15)

STYLE = """<style>
.pw{font-family:system-ui,sans-serif;color:#222;max-width:1250px}
.pw h2{margin:4px 0 2px}.pw .aide{background:#f4f6f8;border-radius:8px;padding:8px 12px;font-size:13px;margin:8px 0}
.pw .niv{display:flex;gap:10px;flex-wrap:wrap;margin:10px 0}
.pw .nv{border:1px solid #ddd;border-radius:10px;padding:8px 10px;width:190px;background:#fff;font-size:13px}
.pw .nv.fini{border-color:#2e7d32;background:#eef7ee}
.pw .barre{display:flex;height:10px;border-radius:5px;overflow:hidden;background:#eee;margin:5px 0}
.pw .cartes{display:flex;flex-wrap:wrap;gap:12px}
.pw .carte{border:1px solid #ccc;border-radius:12px;padding:8px;background:#fff;width:250px;box-shadow:0 1px 4px #0001}
.pw .titre{font-size:13px;color:#555;margin-bottom:6px}
.pw .joueur{display:flex;align-items:center;justify-content:space-between;padding:4px 6px;border-radius:6px;font-size:13px;white-space:nowrap}
.pw .joueur.pwn{background:#e3f0ff}.pw .joueur.sf{background:#f3f3f3}
.pw .pion{display:inline-block;width:13px;height:13px;border-radius:50%;border:1px solid #333;margin-right:6px;vertical-align:-2px}
.pw .reflechit{color:#d35400;font-weight:bold;animation:clig 1s infinite}
@keyframes clig{50%{opacity:.25}}
.pw .pris{font-size:15px;letter-spacing:-2px;color:#555;min-height:18px}
.pw .info{font-size:13px;margin-top:6px}
.pw .mat{height:8px;border-radius:4px;background:#ddd;position:relative;margin-top:4px}
.pw .fin{font-size:13px;padding:3px 0;border-bottom:1px solid #eee}
</style>"""

def case_xy(sq, retourne):
    f, r = chess.square_file(sq), chess.square_rank(sq)
    if retourne:
        return 15 + TAILLE * (7 - f), 15 + TAILLE * r
    return 15 + TAILLE * f, 15 + TAILLE * (7 - r)

def echiquier(bd, coup, retourne, animer):
    svg = chess.svg.board(bd, lastmove=coup, size=250, flipped=retourne, colors=COULEURS,
                          check=bd.king(bd.turn) if bd.is_check() else None)
    if coup and animer:   # la pièce glisse de sa case de départ à sa case d'arrivée
        xa, ya = case_xy(coup.from_square, retourne)
        xb, yb = case_xy(coup.to_square, retourne)
        cible = f'transform="translate({xb}, {yb})" />'
        anim = (f'transform="translate({xb}, {yb})"><animateTransform attributeName="transform" type="translate" '
                f'from="{xa} {ya}" to="{xb} {yb}" dur="0.7s" fill="freeze" /></use>')
        svg = svg.replace(cible, anim, 1)
    return svg

def pieces_prises(bd, couleur):
    # pièces de `couleur` qui ont disparu de l'échiquier
    return "".join(SYMB[couleur][p] * max(0, DEPART[p] - len(bd.pieces(p, couleur))) for p in DEPART)

def materiel(bd, couleur):
    return sum(v * len(bd.pieces(p, couleur)) for p, v in VALEUR.items())

def barre_joueur(nom, classe, couleur_pieces, a_son_tour, prises_adverses):
    pastille = "#fff" if couleur_pieces == chess.WHITE else "#222"
    txt_couleur = "blancs" if couleur_pieces == chess.WHITE else "noirs"
    etat = "<span class='reflechit'>⏳ joue…</span>" if a_son_tour else ""
    return (f"<div class='joueur {classe}'><span><span class='pion' style='background:{pastille}'></span>"
            f"<b>{nom}</b> <small>({txt_couleur})</small></span>{etat}</div>"
            f"<div class='pris' title='pièces capturées'>{prises_adverses}</div>")

def carte(e):
    pwn_blancs = e["blancs"] == JOUEUR
    adv = e["noirs"] if pwn_blancs else e["blancs"]
    c_pwn = chess.WHITE if pwn_blancs else chess.BLACK
    bd = chess.Board(e["fen"])
    coup = chess.Move.from_uci(e["dernier"]) if e["dernier"] else None
    cle = (e["blancs"], e["noirs"], e["ouverture"])
    animer = vu.get(cle) != e["ply"]
    vu[cle] = e["ply"]
    ecart = materiel(bd, c_pwn) - materiel(bd, not c_pwn)
    if ecart > 0:
        mat_txt, mat_coul = f"POWN mène de <b>+{ecart}</b>", "#1e88e5"
    elif ecart < 0:
        mat_txt, mat_coul = f"Stockfish mène de <b>+{-ecart}</b>", "#e53935"
    else:
        mat_txt, mat_coul = "matériel <b>égal</b>", "#999"
    pos = 50 + max(-50, min(50, ecart * 5))
    num = (e["ply"] + 1) // 2
    if coup:
        qui = "POWN" if (e["ply"] % 2 == 1) == pwn_blancs else "Stockfish"
        dernier = f"{num}{'.' if e['ply'] % 2 else '…'} <b>{e['san']}</b> ({qui})"
    else:
        dernier = "–"
    duree = (time.time() - e["debut"]) / 60
    haut = barre_joueur(f"Stockfish {adv[6:]}", "sf", not c_pwn, bd.turn != c_pwn, pieces_prises(bd, c_pwn))
    bas = barre_joueur(JOUEUR, "pwn", c_pwn, bd.turn == c_pwn, pieces_prises(bd, not c_pwn))
    return (f"<div class='carte'><div class='titre'>📖 {e['ouverture']} · coup {num} · {duree:.0f} min</div>"
            f"{haut}{echiquier(bd, coup, not pwn_blancs, animer)}{bas}"
            f"<div class='info'>Dernier coup : {dernier}</div>"
            f"<div class='info'>⚖️ {mat_txt}</div>"
            f"<div class='mat'><div style='position:absolute;left:{pos}%;top:-3px;width:4px;height:14px;background:{mat_coul}'></div>"
            f"<div style='position:absolute;left:50%;top:0;width:1px;height:8px;background:#555'></div></div></div>")

def ecran():
    fait_n = sum(len(v) for v in par_niveau.values())
    total = PAR_NIVEAU * len(STOCKFISH)
    en_cours = []
    for fich in sorted(glob.glob(f"{SUIVI}/*.json")):
        try:
            e = json.load(open(fich))
        except Exception:
            continue
        if "ouverture" in e:   # sinon : partie lancée par une ancienne version du notebook
            en_cours.append(e)
    h = [STYLE, "<div class='pw'>",
         f"<h2>♟️ {JOUEUR} contre Stockfish 17</h2>",
         f"<div>{fait_n}/{total} parties terminées · {len(en_cours)} en cours · {(time.time() - t0) / 60:.0f} min</div>",
         f"<div class='barre' style='width:600px'><div style='width:{100 * fait_n / total:.1f}%;background:#1e88e5'></div></div>",
         "<div class='aide'>Chaque carte = une partie en cours. <b style='color:#1e88e5'>POWN</b> (ton IA) est toujours "
         "<b>en bas</b>, <b>Stockfish</b> en haut. La pastille ⚪/⚫ indique la couleur des pièces. Les cases jaunes = dernier coup. "
         "Sous chaque joueur : les pièces qu'il a capturées. Les niveaux sont joués dans un ordre mélangé exprès.</div>",
         "<div class='niv'>"]
    for sf, lignes in par_niveau.items():
        res = [infos(r)[2] for r in lignes]
        v, n, d = res.count("victoire"), res.count("nulle"), res.count("défaite")
        cours = sum(1 for e in en_cours if sf in (e["blancs"], e["noirs"]))
        sc = f"{100 * (v + n / 2) / len(res):.0f} %" if res else "–"
        fini = " fini" if len(res) == PAR_NIVEAU else ""
        w = lambda x: 100 * x / PAR_NIVEAU
        h.append(f"<div class='nv{fini}'><b>Stockfish {sf[6:]}</b>{' ✔️' if fini else ''}"
                 f"<div class='barre'><div style='width:{w(v)}%;background:#43a047'></div>"
                 f"<div style='width:{w(n)}%;background:#fbc02d'></div><div style='width:{w(d)}%;background:#e53935'></div></div>"
                 f"<small>✅ {v} · 🤝 {n} · ❌ {d} · {len(res)}/{PAR_NIVEAU}"
                 f"{f' · {cours} en cours' if cours else ''}</small><br>Score POWN : <b>{sc}</b></div>")
    h.append("</div><div class='cartes'>")
    for e in en_cours:
        try:
            h.append(carte(e))
        except Exception:
            pass   # fichier d'une ancienne partie (ex. kernel pas redémarré) : on l'ignore
    h.append("</div><h3>Dernières parties terminées</h3>")
    h.append("".join(derniers[-10:][::-1]) or "<i>aucune pour l'instant (une partie dure 10–20 min)</i>")
    h.append("</div>")
    return HTML("".join(h))

def ligne_fin(r):
    adv, couleur, res = infos(r)
    coul = {"victoire": "#43a047", "nulle": "#f9a825", "défaite": "#e53935"}[res]
    return (f"<div class='fin'>{EMOJI[res]} <b style='color:{coul}'>{res.upper()}</b> de POWN ({couleur}) contre "
            f"<b>Stockfish {adv[6:]}</b> · {r['ouverture']} · {r['plies'] // 2} coups · {r['fin'].lower()}</div>")

t0 = time.time()
ecran_id = display(ecran(), display_id=True)
ex = ProcessPoolExecutor(NB_WORKERS, mp_context=mp.get_context("spawn"),
                         initializer=ce.init_worker, initargs=(False, 1, SF, SF_TEMPS))
en_attente = {ex.submit(ce.tache_partie, t + (SUIVI,)) for t in reste}
k = 0
with open(PARTIES, "a") as f:
    while en_attente:
        finis, en_attente = wait(en_attente, timeout=2, return_when=FIRST_COMPLETED)
        for fu in finis:
            k += 1
            try:
                r = fu.result()
            except Exception as e:
                print("erreur :", e)
                continue
            f.write(json.dumps(r) + "\n"); f.flush()
            enregistrer(r)
            adv, couleur, res = infos(r)
            par_niveau[adv].append(r)
            ligne = (f"{EMOJI[res]} contre {adv} ({len(par_niveau[adv])}/{PAR_NIVEAU}) : {r['ouverture']}, "
                     f"POWN avec les {couleur} → {res} ({r['fin']}, {r['plies'] // 2} coups)")
            derniers.append(ligne_fin(r))
            print(f"[{k}/{len(reste)} | {(time.time() - t0) / 60:.0f} min] {ligne}")
            if len(par_niveau[adv]) == PAR_NIVEAU:
                bilan_niveau(adv, par_niveau[adv])
        ecran_id.update(ecran())
        if time.time() - t0 > BUDGET_H * 3600:
            print("budget temps atteint, on s'arrête là")
            for x in en_attente:
                x.cancel()
            break
ex.shutdown(wait=False, cancel_futures=True)

## Résultats

In [ ]:
parties = pd.DataFrame([json.loads(l) for l in open(PARTIES)])
parties = parties.drop_duplicates(["blancs", "noirs", "ouverture"])
parties = parties[parties.blancs.isin(STOCKFISH) | parties.noirs.isin(STOCKFISH)]
parties["adversaire"] = np.where(parties.blancs == JOUEUR, parties.noirs, parties.blancs)
parties["couleur"] = np.where(parties.blancs == JOUEUR, "blancs", "noirs")
gagne = {"1-0": "blancs", "0-1": "noirs"}
parties["resultat_pwn"] = [
    "nulle" if r == "1/2-1/2" else ("victoire" if gagne[r] == c else "défaite")
    for r, c in zip(parties.resultat, parties.couleur)]
parties["score"] = parties.resultat_pwn.map({"victoire": 1.0, "nulle": 0.5, "défaite": 0.0})
with open(f"{SORTIE}/solution_parties.pgn", "w") as f:
    f.write("\n\n".join(parties.pgn))
toutes = []
for _, r in parties.iterrows():
    pos = pd.DataFrame(r.positions)
    pos.insert(0, "adversaire", r.adversaire); pos.insert(1, "ouverture", r.ouverture)
    pos.insert(2, "PWN_joue", r.couleur); pos["resultat_PWN"] = r.resultat_pwn
    toutes.append(pos)
pd.concat(toutes).to_csv(f"{SORTIE}/solution_positions.csv", index=False)
liste = parties[["adversaire", "ouverture", "couleur", "resultat_pwn", "fin", "plies"]].copy()
liste["coups"] = liste.pop("plies") // 2
liste.sort_values(["adversaire", "ouverture", "couleur"]).to_csv(f"{SORTIE}/solution_liste_parties.csv", index=False)

tab = parties.groupby("adversaire").agg(
    victoires=("resultat_pwn", lambda s: (s == "victoire").sum()),
    nulles=("resultat_pwn", lambda s: (s == "nulle").sum()),
    defaites=("resultat_pwn", lambda s: (s == "défaite").sum()),
    parties=("score", "size"),
    score_pct=("score", lambda s: 100 * s.mean()),
    score_blancs=("score", lambda s: 100 * s[parties.loc[s.index, "couleur"] == "blancs"].mean()),
    score_noirs=("score", lambda s: 100 * s[parties.loc[s.index, "couleur"] == "noirs"].mean()),
)
tab["Elo Stockfish"] = tab.index.map(STOCKFISH)
tab = tab.sort_values("Elo Stockfish")
print(len(parties), "parties | fins :", parties.fin.value_counts().to_dict())
print("temps moyen par coup de", JOUEUR, ":",
      round(pd.concat([parties[parties.blancs == JOUEUR].s_par_coup_blancs,
                       parties[parties.noirs == JOUEUR].s_par_coup_noirs]).mean(), 2), "s")
tab.round(1)

In [ ]:
from scipy.optimize import minimize_scalar

def elo_mle(df):
    """Elo de POWN par maximum de vraisemblance, les niveaux Stockfish étant fixés (nulle = ½ victoire)"""
    e_adv, s = df.adversaire.map(STOCKFISH).to_numpy(), df.score.to_numpy()
    def nll(r):
        p = np.clip(1 / (1 + 10 ** ((e_adv - r) / 400)), 1e-9, 1 - 1e-9)
        return -np.sum(s * np.log(p) + (1 - s) * np.log(1 - p))
    return minimize_scalar(nll, bounds=(0, 4000), method="bounded").x

elo = elo_mle(parties)
rng = np.random.default_rng(SEED)
boot = [elo_mle(parties.sample(len(parties), replace=True, random_state=int(rng.integers(1e9)))) for _ in range(500)]
bas, haut = np.percentile(boot, [2.5, 97.5])
print(f"Elo de {JOUEUR} en partie : {elo:.0f}  (IC 95 % : {bas:.0f} – {haut:.0f})")
pd.DataFrame([{"joueur": JOUEUR, "Elo": elo, "IC 95 % bas": bas, "IC 95 % haut": haut, "parties": len(parties)}]) \
  .to_csv(f"{SORTIE}/solution_elo.csv", index=False)
tab.to_csv(f"{SORTIE}/solution_par_niveau.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.linspace(min(NIVEAUX) - 100, max(NIVEAUX) + 100, 200)
ax.plot(x, 100 / (1 + 10 ** ((x - elo) / 400)), color="gray", lw=1, ls="--", label=f"attendu pour Elo {elo:.0f}")
ax.bar(tab["Elo Stockfish"], tab.score_pct, width=120, color="#58a", label="score réel de " + JOUEUR)
for e, v, n in zip(tab["Elo Stockfish"], tab.score_pct, tab.parties):
    ax.text(e, v + 2, f"{v:.0f} %", ha="center", fontsize=9)
ax.axhline(50, color="black", lw=0.5)
ax.axvline(elo, color="#c55", lw=1.5, label=f"Elo estimé {elo:.0f} [{bas:.0f}–{haut:.0f}]")
ax.set_xticks(NIVEAUX); ax.set_xlabel("niveau de Stockfish (UCI_Elo)"); ax.set_ylabel("score de " + JOUEUR + " (%)")
ax.set_ylim(0, 110); ax.set_title(f"{JOUEUR} contre Stockfish 17"); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.savefig(f"{SORTIE}/solution_vs_stockfish.png", dpi=120); plt.show()

### Les fichiers (onglet Output)
| Fichier | Contenu |
|---|---|
| `pown_parties/stockfish_eloXXXX/<ouverture>_PWN-<couleur>.pgn` | chaque partie, à coller sur **lichess.org/paste** pour la rejouer |
| `pown_parties/stockfish_eloXXXX/<ouverture>_PWN-<couleur>_positions.csv` | chaque coup de la partie : qui joue, le coup, la **position (FEN)** obtenue, le temps de réflexion |
| `pown_positions.csv` | toutes les positions de toutes les parties dans un seul fichier |
| `pown_liste_parties.csv` | la liste des parties : adversaire, ouverture, couleur de POWN, résultat, comment ça finit, nombre de coups |
| `pown_par_niveau.csv`, `pown_elo.csv`, `pown_vs_stockfish.png`, `pown_parties.pgn` | les bilans, l'Elo, la courbe, toutes les parties en un PGN |

### Comment lire les résultats
- Le niveau où le score passe sous **50 %**, c'est à peu près l'Elo de POWN@ab3.
- Si POWN fait encore > 50 % contre le niveau le plus fort, ajoute des niveaux plus hauts dans `NIVEAUX` (ex. 2700, 2900) et relance :
  les parties déjà jouées sont gardées.
- Avec 16 parties par niveau (80 au total), l'intervalle fait environ ±70–120 Elo. Pour plus de précision : `N_OUVERTURES = 12` (plus long).
- Beaucoup de fins `MAX_PLIES` ou de nulles par répétition → POWN ne sait pas convertir un avantage en finale.